# 15. Descriptive Stage-Allocation Summary — Herbal Supplements

This notebook converts the Notebook 14 canonical per-case layer into descriptive stage-allocation results. It reports absolute condition means, paired effect estimates, 95% bootstrap intervals, regime summaries, QCHS coverage, QCHS-active diagnostics, and the RetrP contrasts for the learned rerankers. It does not provide the final confirmatory multiplicity decisions.

The primary descriptive endpoint is unconditional NDCG@5 at candidate depth 1,000. The main contrasts are S1-P minus S1-Q, Base minus S1-Q, RankP minus Base, Full minus RankP, RetrP minus Base, Full minus RetrP, and Full minus Base. P-values are deliberately not computed here; the stored significance-schema columns are descriptive placeholders.

The stored identity table requires careful interpretation. On non-cold QCHS-fallback cases, S1-P and S1-Q have identical candidate order. RankP and Full, however, are separate fitted policies on different candidate interfaces. Their final ranks are therefore not required to be identical. The reported LightGBM and Transformer RankP—Full differences on this subset are diagnostic policy differences, not evidence that the Stage 1 fallback failed.

A separate issue remains in the cold stratum. The stored Transformer comparison between Base and Full reports 33 target-rank mismatches and one NDCG@5 mismatch. This contradicts the thesis contract that every reranked condition is exactly identical on cold cases. The notebook still reports `ready_for_thesis_reporting = True` because the combined cold comparison is excluded from its blocking identity subset. That readiness flag therefore does not certify complete cold-condition identity.

The received notebook contains 11 cells: one Markdown cell and 10 executed code cells. Four code cells contain stored outputs and no stored error is present. The execution sequence is mixed rather than a clean top-to-bottom run: counts are 2, 3, 4, 5, 11, 12, 13, 19, 20, and 21.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ==== Imports ====
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math

import numpy as np
import pandas as pd


In [4]:
# ==== Config: Direct Final Notebook 14 Contract ====
NOTEBOOK_NAME = "15_stage_allocation_summary_herbal.ipynb"
CATEGORY_ID = "herbal"
CATEGORY_KEY = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"

PROJECT_ROOT = Path(f"/content/drive/MyDrive/thesis_recsys/categories/{CATEGORY_KEY}")
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
ANALYSIS_DIR = OUTPUTS_DIR / "analysis"
OUT_DIR = ANALYSIS_DIR / "stage_allocation_summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_AGGREGATE_DIR = OUTPUTS_DIR / "pipeline_aggregate"
PIPELINE_MANIFEST_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_manifest.json"
CANONICAL_RAW_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_canonical_per_case_metrics.parquet"
CANONICAL_PRIMARY_PATH = (
    PIPELINE_AGGREGATE_DIR
    / "pipeline_canonical_primary_ndcg5_depth1000.parquet"
)
NB14_UPSTREAM_READINESS_PATH = (
    PIPELINE_AGGREGATE_DIR / "pipeline_upstream_readiness.csv"
)
NB14_SOURCE_HASHES_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_source_hashes.csv"
NB14_OVERALL_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_results_overall.csv"
NB14_BY_REGIME_POOL_DEPTH_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_results_by_regime_pool_depth.csv"
NB14_BY_REGIME_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_results_by_regime.csv"
NB14_BY_POOL_DEPTH_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_results_by_pool_depth.csv"
NB14_FIVE_CONDITION_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_five_condition_comparison.csv"
NB14_AVAILABILITY_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_method_depth_availability.csv"
NB14_SOURCE_INVENTORY_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_source_inventory.csv"

CANONICAL_CONDITIONS = ["P0", "P1-only", "P2-Q", "P2-P", "b02", "Full"]
RERANKER_FAMILIES = ["shannon", "lightgbm", "gam", "transformer"]
METRIC_NAMES = ["HitRate", "NDCG", "MRR"]
RANDOM_SEED = 20250320
N_BOOTSTRAP = 2000
N_SIGN_FLIP = 0
RESAMPLE_CHUNK_SIZE = 128
ALPHA = 0.05

REPORT_POOL_DEPTH = 1000  # report depth 1000 (2026-07-23); reranking always top-1000; all-depth panels unchanged
PRIMARY_METRIC_NAME = "NDCG"
PRIMARY_METRIC_CUTOFF = 5
PRIMARY_RERANKER_FAMILIES = ["lightgbm", "transformer"]

OUTPUT_FILES = {
    "overall_by_reranker": OUT_DIR / "stage_allocation_overall_by_reranker.csv",
    "by_regime_reranker": OUT_DIR / "stage_allocation_by_regime_reranker.csv",
    "by_regime_pool_depth": OUT_DIR / "stage_allocation_by_regime_pool_depth.csv",
    "by_pool_depth": OUT_DIR / "stage_allocation_by_pool_depth.csv",
    "five_condition_comparison": OUT_DIR / "stage_allocation_five_condition_comparison.csv",
    "paired_contrasts": OUT_DIR / "stage_allocation_paired_contrasts.csv",
    "significance_tests": OUT_DIR / "stage_allocation_significance_tests.csv",
    "qchs_coverage": OUT_DIR / "stage_allocation_qchs_coverage.csv",
    "qchs_active_subset": OUT_DIR / "stage_allocation_qchs_active_subset.csv",
    "method_availability": OUT_DIR / "stage_allocation_method_availability.csv",
    "stage_delta_per_case": OUT_DIR / "stage_delta_per_case.parquet",
    "stage_delta_paired_summary": OUT_DIR / "stage_delta_paired_summary.csv",
    "cold_fallback_identity_qc": OUT_DIR / "cold_fallback_identity_qc.csv",
    "aggregate_metric_authority_qc": OUT_DIR / "aggregate_metric_authority_qc.csv",
    "manifest": OUT_DIR / "stage_allocation_manifest.json",
}


In [5]:
# ==== Validation, Display, and Descriptive Paired-Effect Helpers ====
def require_columns(df, required, label):
    missing = sorted(set(required).difference(df.columns))
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def require_unique_columns(df, label):
    duplicated = df.columns[df.columns.duplicated()].astype(str).tolist()
    if duplicated:
        raise RuntimeError(f"{label} has duplicated output columns: {duplicated}")



def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def boolean_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean")
    text = series.astype("string").str.strip().str.lower()
    numeric = pd.to_numeric(series, errors="coerce")
    mapped = text.map({
        "true": True, "false": False, "yes": True, "no": False,
        "1": True, "0": False,
    })
    return mapped.where(mapped.notna(), numeric.map({1.0: True, 0.0: False})).astype("boolean")


def add_category_columns(df):
    out = df.copy()
    out.insert(0, "category_label", CATEGORY_LABEL)
    out.insert(0, "category_key", CATEGORY_KEY)
    out.insert(0, "category_id", CATEGORY_ID)
    return out


def repeat_shared_summary_for_display(df):
    require_columns(df, ["stage_condition", "method_family", "reranker_method"], "Notebook 14 summary")
    shared = df.loc[df["stage_condition"].isin(["P0", "P1-only"])].copy()
    stage2 = df.loc[df["stage_condition"].isin(["P2-Q", "P2-P", "b02", "Full"])].copy()
    repeated = []
    for family in RERANKER_FAMILIES:
        part = shared.copy()
        part["reranker_family"] = family
        part["shared_baseline_repeated_for_display"] = True
        repeated.append(part)
    stage2["reranker_family"] = stage2["method_family"].astype(str)
    stage2["shared_baseline_repeated_for_display"] = False
    out = pd.concat([*repeated, stage2], ignore_index=True, sort=False)
    out["stage_condition"] = pd.Categorical(
        out["stage_condition"], categories=CANONICAL_CONDITIONS, ordered=True
    )
    out = out.sort_values(
        [
            column for column in [
                "reranker_family", "stage_condition", "regime",
                "candidate_pool_depth", "metric_name", "metric_cutoff",
            ] if column in out.columns
        ],
        kind="mergesort",
    ).reset_index(drop=True)
    out["stage_condition"] = out["stage_condition"].astype(str)
    return add_category_columns(out)


def stable_seed(*parts):
    payload = "|".join(map(str, (RANDOM_SEED, *parts))).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little") % (2**32 - 1)


def paired_resampling_statistics(delta_values, seed_parts):
    delta = pd.to_numeric(pd.Series(delta_values), errors="raise").to_numpy(dtype=float)
    if not np.isfinite(delta).all() or len(delta) == 0:
        raise RuntimeError("Paired deltas must be finite and non-empty.")
    n = len(delta)
    observed_mean = float(delta.mean())
    observed_median = float(np.median(delta))

    bootstrap_rng = np.random.default_rng(stable_seed("bootstrap", *seed_parts))
    bootstrap_means = np.empty(N_BOOTSTRAP, dtype=float)
    for start in range(0, N_BOOTSTRAP, RESAMPLE_CHUNK_SIZE):
        stop = min(start + RESAMPLE_CHUNK_SIZE, N_BOOTSTRAP)
        indices = bootstrap_rng.integers(0, n, size=(stop - start, n))
        bootstrap_means[start:stop] = delta[indices].mean(axis=1)
    ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])

    # Confirmatory sign-flip inference is computed only in Notebook 16.
    p_value = np.nan

    win_count = int(np.count_nonzero(delta > 0.0))
    tie_count = int(np.count_nonzero(delta == 0.0))
    loss_count = int(np.count_nonzero(delta < 0.0))
    return {
        "paired_query_count": int(n),
        "mean_difference": observed_mean,
        "median_difference": observed_median,
        "bootstrap_ci_95_low": float(ci_low),
        "bootstrap_ci_95_high": float(ci_high),
        "sign_flip_p_value": p_value,
        "win_count": win_count,
        "tie_count": tie_count,
        "loss_count": loss_count,
        "positive_delta_rate": float(win_count / n),
        "bootstrap_resamples": int(N_BOOTSTRAP),
        "sign_flip_permutations": int(N_SIGN_FLIP),
        "random_seed": int(stable_seed(*seed_parts)),
    }


def condition_rows(raw_df, condition, family):
    if condition in {"P0", "P1-only"}:
        out = raw_df.loc[
            raw_df["stage_condition"].eq(condition)
            & raw_df["reranker_method"].eq("none")
            & raw_df["method_family"].eq("shared_retrieval_baseline")
        ].copy()
    else:
        out = raw_df.loc[
            raw_df["stage_condition"].eq(condition)
            & raw_df["method_family"].eq(family)
        ].copy()
    key = ["case_id", "candidate_pool_depth", "metric_name", "metric_cutoff"]
    if out.duplicated(key).any():
        raise RuntimeError(f"Canonical rows are duplicated for condition={condition}, family={family}.")
    return out


def build_paired_contrast(raw_df, contrast_name, left_condition, right_condition, family, subset_mode="all_cases"):
    keep = [
        "case_id", "regime", "candidate_pool_depth", "metric_name", "metric_cutoff",
        "metric_value", "qchs_profile_available", "profile_fallback_flag",
    ]
    left = condition_rows(raw_df, left_condition, family)[keep].copy()
    right = condition_rows(raw_df, right_condition, family)[keep].copy()
    combo_cols = ["candidate_pool_depth", "metric_name", "metric_cutoff"]
    left_combos = set(map(tuple, left[combo_cols].drop_duplicates().to_numpy()))
    right_combos = set(map(tuple, right[combo_cols].drop_duplicates().to_numpy()))
    common_combos = sorted(left_combos.intersection(right_combos))

    rows = []
    for depth, metric_name, metric_cutoff in common_combos:
        left_part = left.loc[
            left["candidate_pool_depth"].eq(depth)
            & left["metric_name"].eq(metric_name)
            & left["metric_cutoff"].eq(metric_cutoff)
        ].copy()
        right_part = right.loc[
            right["candidate_pool_depth"].eq(depth)
            & right["metric_name"].eq(metric_name)
            & right["metric_cutoff"].eq(metric_cutoff)
        ].copy()
        left_cases = set(left_part["case_id"].astype(str))
        right_cases = set(right_part["case_id"].astype(str))
        if left_cases != right_cases:
            raise RuntimeError(
                "Paired query universe mismatch: "
                f"contrast={contrast_name}, family={family}, depth={depth}, "
                f"metric={metric_name}@{metric_cutoff}, "
                f"left_only={len(left_cases - right_cases)}, right_only={len(right_cases - left_cases)}"
            )
        paired = left_part.merge(
            right_part,
            on=["case_id", "candidate_pool_depth", "metric_name", "metric_cutoff"],
            how="inner",
            suffixes=("_left", "_right"),
            validate="one_to_one",
        )
        if not paired["regime_left"].astype(str).eq(paired["regime_right"].astype(str)).all():
            raise RuntimeError(f"Regime mismatch in paired contrast: {contrast_name}")
        paired["regime"] = paired["regime_right"].astype(str)
        paired["delta"] = (
            pd.to_numeric(paired["metric_value_right"], errors="raise")
            - pd.to_numeric(paired["metric_value_left"], errors="raise")
        )
        if subset_mode == "qchs_active":
            active = boolean_series(paired["qchs_profile_available_right"]).fillna(False)
            paired = paired.loc[active].copy()
        if paired.empty:
            continue

        scopes = [("overall", "overall", paired)]
        scopes.extend(
            ("regime", str(regime), sub)
            for regime, sub in paired.groupby("regime", dropna=False, observed=True)
        )
        for user_scope, regime, scope_df in scopes:
            seed_parts = (
                CATEGORY_ID, contrast_name, family, depth,
                metric_name, metric_cutoff, user_scope, regime, subset_mode,
            )
            stats = paired_resampling_statistics(scope_df["delta"], seed_parts)
            rows.append({
                "category_id": CATEGORY_ID,
                "category_key": CATEGORY_KEY,
                "category_label": CATEGORY_LABEL,
                "contrast_name": contrast_name,
                "left_condition": left_condition,
                "right_condition": right_condition,
                "reranker_family": family,
                "candidate_pool_depth": int(depth),
                "metric_name": metric_name,
                "metric_cutoff": int(metric_cutoff),
                "user_scope": user_scope,
                "regime": regime,
                "analysis_subset": subset_mode,
                "primary_result": subset_mode == "all_cases",
                "includes_qchs_fallback_cases": subset_mode == "all_cases",
                **stats,
            })
    return pd.DataFrame(rows)



In [11]:
# ==== Load and Validate Final Notebook 14 Canonical Artifacts ====
required_input_paths = [
    PIPELINE_MANIFEST_PATH,
    CANONICAL_RAW_PATH,
    CANONICAL_PRIMARY_PATH,
    NB14_UPSTREAM_READINESS_PATH,
    NB14_SOURCE_HASHES_PATH,
    NB14_OVERALL_PATH,
    NB14_BY_REGIME_POOL_DEPTH_PATH,
    NB14_BY_REGIME_PATH,
    NB14_BY_POOL_DEPTH_PATH,
    NB14_FIVE_CONDITION_PATH,
    NB14_AVAILABILITY_PATH,
    NB14_SOURCE_INVENTORY_PATH,
]
missing_inputs = [str(path) for path in required_input_paths if not path.exists()]
if missing_inputs:
    raise RuntimeError(f"Missing final Notebook 14 canonical artifacts: {missing_inputs}")

pipeline_manifest = json.loads(PIPELINE_MANIFEST_PATH.read_text(encoding="utf-8"))
canonical_raw_df = pd.read_parquet(CANONICAL_RAW_PATH)
canonical_primary_df = pd.read_parquet(CANONICAL_PRIMARY_PATH)
nb14_upstream_readiness_df = pd.read_csv(NB14_UPSTREAM_READINESS_PATH)
nb14_source_hashes_df = pd.read_csv(NB14_SOURCE_HASHES_PATH)
nb14_overall_df = pd.read_csv(NB14_OVERALL_PATH)
nb14_by_regime_pool_depth_df = pd.read_csv(NB14_BY_REGIME_POOL_DEPTH_PATH)
nb14_by_regime_df = pd.read_csv(NB14_BY_REGIME_PATH)
nb14_by_pool_depth_df = pd.read_csv(NB14_BY_POOL_DEPTH_PATH)
nb14_five_condition_df = pd.read_csv(NB14_FIVE_CONDITION_PATH)
nb14_availability_df = pd.read_csv(NB14_AVAILABILITY_PATH)
nb14_source_inventory_df = pd.read_csv(NB14_SOURCE_INVENTORY_PATH)

canonical_required = [
    "case_id", "query_id", "user_id", "regime",
    "stage_condition", "reranker_method", "method_family",
    "candidate_pool_depth", "metric_name", "metric_cutoff", "metric_value",
    "target_exposed", "target_rank", "qchs_profile_available",
    "profile_fallback_flag", "candidate_source", "source_notebook",
    "shared_stage1_baseline", "shared_baseline_repeated_for_display",
]
require_columns(canonical_raw_df, canonical_required, "Notebook 14 canonical raw data")
require_unique_columns(canonical_raw_df, "Notebook 14 canonical raw data")

canonical_key = [
    "case_id", "stage_condition", "reranker_method",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
]
if canonical_raw_df.duplicated(canonical_key).any():
    raise RuntimeError("Notebook 14 canonical raw data contain duplicated inferential keys.")

primary_required = [
    "case_id", "stage_condition", "analysis_method_family",
    "candidate_pool_depth", "metric_name", "metric_cutoff", "metric_value",
]
require_columns(
    canonical_primary_df, primary_required,
    "Notebook 14 locked primary method grid",
)
require_unique_columns(
    canonical_primary_df, "Notebook 14 locked primary method grid"
)
primary_grid_key = [
    "case_id", "stage_condition", "analysis_method_family",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
]
if canonical_primary_df.duplicated(primary_grid_key).any():
    raise RuntimeError("Notebook 14 locked primary method grid has duplicate keys.")
if not canonical_primary_df["candidate_pool_depth"].eq(REPORT_POOL_DEPTH).all():
    raise RuntimeError("Notebook 14 primary grid contains a non-headline depth.")
if not canonical_primary_df["metric_name"].eq(PRIMARY_METRIC_NAME).all():
    raise RuntimeError("Notebook 14 primary grid contains a non-NDCG metric.")
if not canonical_primary_df["metric_cutoff"].eq(PRIMARY_METRIC_CUTOFF).all():
    raise RuntimeError("Notebook 14 primary grid contains a non-5 cutoff.")

_primary_cases = canonical_primary_df["case_id"].astype(str).nunique()

DESIGN_EXCLUDED_METHOD_CONDITION_PAIRS = {
    (str(_entry.get("reranker_family", "")), str(_entry.get("stage_condition", "")))
    for _entry in (
        pipeline_manifest.get("structural_unavailable_cells", {})
        .get("design_excluded_method_condition_pairs", [])
    )
}

DESIGN_METHOD_CONDITION_PAIRS = {
    (_family, _condition)
    for _family in RERANKER_FAMILIES
    for _condition in CANONICAL_CONDITIONS
}.difference(DESIGN_EXCLUDED_METHOD_CONDITION_PAIRS)

_observed_primary_pairs = set(
    canonical_primary_df[["analysis_method_family", "stage_condition"]]
    .astype(str)
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

if _observed_primary_pairs != DESIGN_METHOD_CONDITION_PAIRS:
    raise RuntimeError(
        "Notebook 14 locked primary method grid does not match the design registry: "
        f"missing={sorted(DESIGN_METHOD_CONDITION_PAIRS - _observed_primary_pairs)}, "
        f"extra={sorted(_observed_primary_pairs - DESIGN_METHOD_CONDITION_PAIRS)}"
    )

_expected_primary_rows = _primary_cases * len(DESIGN_METHOD_CONDITION_PAIRS)
notebook14_primary_grid_complete = bool(len(canonical_primary_df) == _expected_primary_rows)
if not notebook14_primary_grid_complete:
    raise RuntimeError(
        "Notebook 14 locked primary method grid is incomplete: "
        f"expected={_expected_primary_rows}, observed={len(canonical_primary_df)}"
    )

print(
    "Primary method grid contract:",
    f"{_primary_cases} cases x {len(DESIGN_METHOD_CONDITION_PAIRS)} design cells "
    f"= {_expected_primary_rows} rows; "
    f"design-excluded={sorted(DESIGN_EXCLUDED_METHOD_CONDITION_PAIRS)}"
)

observed_conditions = set(canonical_raw_df["stage_condition"].astype(str))
if observed_conditions != set(CANONICAL_CONDITIONS):
    raise RuntimeError(f"Notebook 14 condition contract mismatch: {sorted(observed_conditions)}")
stage2_families = set(
    canonical_raw_df.loc[
        canonical_raw_df["stage_condition"].isin(["P2-Q", "P2-P", "b02", "Full"]),
        "method_family",
    ].astype(str)
)
if stage2_families != set(RERANKER_FAMILIES):
    raise RuntimeError(f"Notebook 14 reranker-family contract mismatch: {sorted(stage2_families)}")

shared_raw = canonical_raw_df.loc[
    canonical_raw_df["stage_condition"].isin(["P0", "P1-only"])
]
if set(shared_raw["reranker_method"]) != {"none"}:
    raise RuntimeError("P0/P1 must remain one shared inferential source with reranker_method='none'.")
if shared_raw["shared_baseline_repeated_for_display"].astype(bool).any():
    raise RuntimeError("Repeated P0/P1 presentation rows entered Notebook 14 inferential data.")

gam_depths = set(
    canonical_raw_df.loc[
        canonical_raw_df["method_family"].eq("gam")
        & canonical_raw_df["stage_condition"].isin(["P2-Q", "P2-P", "b02", "Full"]),
        "candidate_pool_depth",
    ].astype(int)
)
expected_gam_depths = set(
    int(depth) for depth in pipeline_manifest["pool_depth_contract"]["candidate_pool_depths"]
)
if gam_depths != expected_gam_depths:
    raise RuntimeError(
        f"GAM fixed-prefix depths disagree with Notebook 14: "
        f"expected={sorted(expected_gam_depths)}, found={sorted(gam_depths)}"
    )
if pipeline_manifest.get("run_status") != "SUCCESS":
    raise RuntimeError("Notebook 14 canonical aggregation is not sealed with run_status=SUCCESS.")
if pipeline_manifest.get("ready_for_downstream") is not True:
    raise RuntimeError("Notebook 14 is not sealed as ready_for_downstream.")
if pipeline_manifest.get("full_transfer_readiness") is not True:
    raise RuntimeError("Notebook 14 Full transfer readiness failed.")
_notebook14_canonical_raw_sha256 = file_sha256(CANONICAL_RAW_PATH)
_notebook14_canonical_primary_sha256 = file_sha256(CANONICAL_PRIMARY_PATH)
notebook14_hashes_verified = bool(
    pipeline_manifest.get("canonical_raw_sha256") == _notebook14_canonical_raw_sha256
    and pipeline_manifest.get("canonical_primary_sha256") == _notebook14_canonical_primary_sha256
)
if pipeline_manifest.get("canonical_raw_sha256") != _notebook14_canonical_raw_sha256:
    raise RuntimeError("Notebook 14 canonical raw SHA does not match its manifest.")
if pipeline_manifest.get("canonical_primary_sha256") != _notebook14_canonical_primary_sha256:
    raise RuntimeError("Notebook 14 primary-grid SHA does not match its manifest.")
if (
    nb14_upstream_readiness_df.empty
    or not nb14_upstream_readiness_df["gate_status"].astype(str).eq("SUCCESS").all()
):
    raise RuntimeError("Notebook 14 upstream readiness table is incomplete or failed.")
if nb14_source_hashes_df.empty:
    raise RuntimeError("Notebook 14 source-hash table is empty.")

manifest_outputs = pipeline_manifest.get("output_paths", {})
if Path(manifest_outputs.get("canonical_raw", "")).name != CANONICAL_RAW_PATH.name:
    raise RuntimeError("Notebook 14 manifest canonical-raw path disagrees with the fixed contract.")

stage_allocation_overall_by_reranker_df = repeat_shared_summary_for_display(nb14_overall_df)
stage_allocation_by_regime_pool_depth_df = repeat_shared_summary_for_display(
    nb14_by_regime_pool_depth_df
)
stage_allocation_by_regime_reranker_df = repeat_shared_summary_for_display(
    nb14_by_regime_df
)
stage_allocation_by_pool_depth_df = repeat_shared_summary_for_display(
    nb14_by_pool_depth_df
)

require_columns(
    nb14_five_condition_df,
    ["stage_condition", "display_reranker_family", "shared_baseline_repeated_for_display"],
    "Notebook 14 five-condition comparison",
)
stage_allocation_five_condition_comparison_df = add_category_columns(
    nb14_five_condition_df.rename(
        columns={"display_reranker_family": "reranker_family"}
    )
)
stage_allocation_method_availability_df = add_category_columns(nb14_availability_df)


Primary method grid contract: 1968 cases x 23 design cells = 45264 rows; design-excluded=[('shannon', 'b02')]


In [12]:
# ==== Paired Stage Contrasts from the non-duplicated Canonical Raw Table ====
contrast_frames = [
    build_paired_contrast(
        canonical_raw_df,
        "P1-only_minus_P0",
        "P0",
        "P1-only",
        "shared_stage1",
    )
]
for family in RERANKER_FAMILIES:
    contrast_frames.extend([
        build_paired_contrast(
            canonical_raw_df,
            "P2-Q_minus_P0",
            "P0",
            "P2-Q",
            family,
        ),
        build_paired_contrast(
            canonical_raw_df,
            "P2-P_minus_P2-Q",
            "P2-Q",
            "P2-P",
            family,
        ),
        build_paired_contrast(
            canonical_raw_df,
            "Full_minus_P2-P",
            "P2-P",
            "Full",
            family,
        ),
        build_paired_contrast(
            canonical_raw_df,
            "Full_minus_P2-Q",
            "P2-Q",
            "Full",
            family,
        ),
    ])
    if family != "shannon":  # b02 (S1-P no-prior); shannon b02 = S1-P raw label row, added downstream
        contrast_frames.extend([
            build_paired_contrast(canonical_raw_df, "b02_minus_P2-Q", "P2-Q", "b02", family),
            build_paired_contrast(canonical_raw_df, "Full_minus_b02", "b02", "Full", family),
        ])

stage_allocation_paired_contrasts_df = pd.concat(
    contrast_frames, ignore_index=True, sort=False
)
if (
    stage_allocation_paired_contrasts_df.loc[
        stage_allocation_paired_contrasts_df["contrast_name"].eq("P1-only_minus_P0"),
        "reranker_family",
    ].unique().tolist() != ["shared_stage1"]
):
    raise RuntimeError("The shared P1-only minus P0 contrast was repeated by reranker family.")

stage_allocation_paired_contrasts_df["holm_family_id"] = "not_applicable"
stage_allocation_paired_contrasts_df["sign_flip_p_value_holm"] = np.nan
stage_allocation_paired_contrasts_df["holm_reject_at_0_05"] = False
stage_allocation_paired_contrasts_df["multiplicity_method"] = (
    "not_applied_descriptive_only"
)
stage_allocation_paired_contrasts_df["alpha"] = np.nan

significance_columns = [
    "category_id", "category_key", "category_label", "contrast_name",
    "left_condition", "right_condition", "reranker_family",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
    "user_scope", "regime", "analysis_subset", "paired_query_count",
    "mean_difference", "bootstrap_ci_95_low", "bootstrap_ci_95_high",
    "sign_flip_p_value", "sign_flip_p_value_holm",
    "holm_reject_at_0_05", "holm_family_id", "multiplicity_method",
    "alpha", "random_seed",
]
stage_allocation_significance_tests_df = stage_allocation_paired_contrasts_df[
    significance_columns
].copy()
stage_allocation_significance_tests_df["inference_role"] = "not_computed_notebook16_is_inference_authority"
stage_allocation_significance_tests_df["confirmatory_inference_authority"] = False
stage_allocation_paired_contrasts_df["inference_role"] = "descriptive_effect_size_only"
stage_allocation_paired_contrasts_df["confirmatory_inference_authority"] = False


In [13]:
# ==== QCHS Coverage and Conditional active-only Diagnostics ====
p1_case_contract_df = (
    canonical_raw_df.loc[canonical_raw_df["stage_condition"].eq("P1-only")]
    [
        [
            "case_id", "regime", "qchs_profile_available",
            "profile_fallback_flag",
        ]
    ]
    .drop_duplicates()
)
if p1_case_contract_df.duplicated("case_id").any():
    raise RuntimeError("QCHS contract fields vary across P1-only metric rows.")
p1_case_contract_df["qchs_profile_available"] = boolean_series(
    p1_case_contract_df["qchs_profile_available"]
)
p1_case_contract_df["profile_fallback_flag"] = boolean_series(
    p1_case_contract_df["profile_fallback_flag"]
)
if (
    p1_case_contract_df["qchs_profile_available"].fillna(False)
    & p1_case_contract_df["profile_fallback_flag"].fillna(False)
).any():
    raise RuntimeError("A case cannot be both QCHS-active and baseline-equivalent fallback.")

coverage_rows = []
coverage_scopes = [("overall", "overall", p1_case_contract_df)]
coverage_scopes.extend(
    ("regime", str(regime), sub)
    for regime, sub in p1_case_contract_df.groupby("regime", dropna=False, observed=True)
)
for user_scope, regime, sub in coverage_scopes:
    total = int(sub["case_id"].nunique())
    active = int(sub.loc[sub["qchs_profile_available"].fillna(False), "case_id"].nunique())
    fallback = int(sub.loc[sub["profile_fallback_flag"].fillna(False), "case_id"].nunique())
    coverage_rows.append({
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_label": CATEGORY_LABEL,
        "user_scope": user_scope,
        "regime": regime,
        "total_cases": total,
        "qchs_active_cases": active,
        "baseline_equivalent_fallback_cases": fallback,
        "qchs_profile_coverage_rate": float(active / total) if total else np.nan,
        "fallback_rate": float(fallback / total) if total else np.nan,
    })
stage_allocation_qchs_coverage_df = pd.DataFrame(coverage_rows)

qchs_active_frames = [
    build_paired_contrast(
        canonical_raw_df,
        "P1-only_minus_P0",
        "P0",
        "P1-only",
        "shared_stage1",
        subset_mode="qchs_active",
    )
]
for family in RERANKER_FAMILIES:
    qchs_active_frames.extend([
        build_paired_contrast(
            canonical_raw_df,
            "Full_minus_P2-P",
            "P2-P",
            "Full",
            family,
            subset_mode="qchs_active",
        ),
        build_paired_contrast(
            canonical_raw_df,
            "Full_minus_P2-Q",
            "P2-Q",
            "Full",
            family,
            subset_mode="qchs_active",
        ),
    ])
stage_allocation_qchs_active_subset_df = pd.concat(
    qchs_active_frames, ignore_index=True, sort=False
)
stage_allocation_qchs_active_subset_df["diagnostic_label"] = (
    "conditional_QCHS_active_only_not_primary"
)


In [19]:
# ==== Primary NDCG@5 Stage Deltas and Metric Authority QC ====
def ndcg_from_rank(target_rank, cutoff):
    rank = pd.to_numeric(target_rank, errors="coerce").astype(float)
    valid = rank.notna() & rank.le(int(cutoff))
    rank_for_metric = rank.fillna(np.inf)
    return pd.Series(
        np.where(valid, 1.0 / np.log2(rank_for_metric + 1.0), 0.0),
        index=rank.index,
        dtype=float,
    )


def primary_condition_rows(condition, family):
    if condition in {"P0", "P1-only"}:
        rows = primary_metric_df.loc[
            primary_metric_df["stage_condition"].eq(condition)
            & primary_metric_df["reranker_method"].eq("none")
            & primary_metric_df["method_family"].eq("shared_retrieval_baseline")
        ].copy()
    else:
        rows = primary_metric_df.loc[
            primary_metric_df["stage_condition"].eq(condition)
            & primary_metric_df["method_family"].eq(family)
        ].copy()
    if rows.duplicated("case_id").any():
        raise RuntimeError(
            f"Primary metric rows are duplicated for condition={condition}, family={family}."
        )
    return rows


def build_primary_case_delta(contrast):
    family = contrast["reranker_family"]
    left = primary_condition_rows(contrast["left_condition"], family)
    right = primary_condition_rows(contrast["right_condition"], family)
    paired = left.merge(
        right,
        on="case_id",
        how="outer",
        suffixes=("_left", "_right"),
        indicator=True,
        validate="one_to_one",
    )
    if not paired["_merge"].eq("both").all():
        counts = paired["_merge"].value_counts().to_dict()
        raise RuntimeError(
            f"Primary paired universe mismatch for {contrast['contrast_name']}: {counts}"
        )
    for column in ["query_id", "user_id", "regime"]:
        if not paired[f"{column}_left"].astype(str).eq(
            paired[f"{column}_right"].astype(str)
        ).all():
            raise RuntimeError(
                f"{column} mismatch in primary paired contrast: {contrast['contrast_name']}"
            )

    out = pd.DataFrame({
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_label": CATEGORY_LABEL,
        "case_id": paired["case_id"].astype(str),
        "query_id": paired["query_id_right"].astype(str),
        "user_id": paired["user_id_right"].astype(str),
        "regime": paired["regime_right"].astype(str),
        "contrast_name": contrast["contrast_name"],
        "effect_label": contrast["effect_label"],
        "left_condition": contrast["left_condition"],
        "right_condition": contrast["right_condition"],
        "reranker_family": family,
        "reranker_role": contrast["reranker_role"],
        "candidate_pool_depth": REPORT_POOL_DEPTH,
        "metric_name": PRIMARY_METRIC_NAME,
        "metric_cutoff": PRIMARY_METRIC_CUTOFF,
        "metric_value_left": pd.to_numeric(
            paired["metric_value_left"], errors="raise"
        ).astype(float),
        "metric_value_right": pd.to_numeric(
            paired["metric_value_right"], errors="raise"
        ).astype(float),
        "target_rank_left": pd.to_numeric(
            paired["target_rank_left"], errors="coerce"
        ).astype("Int64"),
        "target_rank_right": pd.to_numeric(
            paired["target_rank_right"], errors="coerce"
        ).astype("Int64"),
        "candidate_source_left": paired["candidate_source_left"].astype(str),
        "candidate_source_right": paired["candidate_source_right"].astype(str),
        "source_notebook_left": paired["source_notebook_left"].astype(str),
        "source_notebook_right": paired["source_notebook_right"].astype(str),
        "qchs_profile_available": boolean_series(
            paired["qchs_profile_available_right"]
        ).fillna(False).astype(bool),
        "profile_fallback_flag": boolean_series(
            paired["profile_fallback_flag_right"]
        ).fillna(False).astype(bool),
    })
    out["delta_ndcg_at_5"] = out["metric_value_right"] - out["metric_value_left"]
    out["is_strong"] = out["regime"].str.lower().eq("strong")
    out["is_non_cold"] = ~out["regime"].str.lower().eq("cold")
    out["is_qchs_active"] = out["qchs_profile_available"]
    return out


def user_cluster_bootstrap_statistics(delta_frame, seed_parts):
    values = pd.to_numeric(delta_frame["delta_ndcg_at_5"], errors="raise")
    if values.empty or not np.isfinite(values.to_numpy(dtype=float)).all():
        raise RuntimeError("Primary paired deltas must be finite and non-empty.")
    cluster = (
        delta_frame.assign(delta_value=values)
        .groupby("user_id", dropna=False, observed=True)
        .agg(delta_sum=("delta_value", "sum"), case_count=("case_id", "size"))
        .reset_index(drop=True)
    )
    cluster_sum = cluster["delta_sum"].to_numpy(dtype=float)
    cluster_count = cluster["case_count"].to_numpy(dtype=float)
    cluster_n = len(cluster)
    rng = np.random.default_rng(stable_seed("user_cluster_bootstrap", *seed_parts))
    bootstrap_means = np.empty(N_BOOTSTRAP, dtype=float)
    for start in range(0, N_BOOTSTRAP, RESAMPLE_CHUNK_SIZE):
        stop = min(start + RESAMPLE_CHUNK_SIZE, N_BOOTSTRAP)
        indices = rng.integers(0, cluster_n, size=(stop - start, cluster_n))
        bootstrap_means[start:stop] = (
            cluster_sum[indices].sum(axis=1) / cluster_count[indices].sum(axis=1)
        )
    ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])
    delta = values.to_numpy(dtype=float)
    return {
        "paired_case_count": int(len(delta_frame)),
        "paired_user_count": int(cluster_n),
        "mean_delta_ndcg_at_5": float(delta.mean()),
        "median_delta_ndcg_at_5": float(np.median(delta)),
        "bootstrap_ci_95_low": float(ci_low),
        "bootstrap_ci_95_high": float(ci_high),
        "positive_delta_rate": float(np.mean(delta > 0.0)),
        "zero_delta_rate": float(np.mean(delta == 0.0)),
        "negative_delta_rate": float(np.mean(delta < 0.0)),
        "bootstrap_resamples": int(N_BOOTSTRAP),
        "resampling_unit": "user_id cluster",
        "random_seed": int(stable_seed("user_cluster_bootstrap", *seed_parts)),
    }


primary_metric_df = canonical_raw_df.loc[
    canonical_raw_df["candidate_pool_depth"].eq(REPORT_POOL_DEPTH)
    & canonical_raw_df["metric_name"].eq(PRIMARY_METRIC_NAME)
    & canonical_raw_df["metric_cutoff"].eq(PRIMARY_METRIC_CUTOFF)
].copy()
if primary_metric_df.empty:
    raise RuntimeError("No canonical rows match the primary NDCG@5 depth-1000 contract.")

case_query_grain = primary_metric_df[["case_id", "query_id"]].drop_duplicates()
case_user_grain = primary_metric_df[["case_id", "user_id"]].drop_duplicates()
if case_query_grain.duplicated("case_id").any():
    raise RuntimeError("case_id maps to more than one query_id in the primary artifact.")
if case_user_grain.duplicated("case_id").any():
    raise RuntimeError("case_id maps to more than one user_id in the primary artifact.")
if case_query_grain["query_id"].isna().any() or case_user_grain["user_id"].isna().any():
    raise RuntimeError("Primary case/query/user identifiers must not be missing.")

STAGE_DELTA_CONTRASTS = [
    {
        "contrast_name": "P1-only_minus_P0",
        "left_condition": "P0",
        "right_condition": "P1-only",
        "reranker_family": "shared_stage1",
        "reranker_role": "shared_stage1",
        "effect_label": "Stage 1 personalization effect",
    },
]
for family, role in [("lightgbm", "primary"), ("transformer", "secondary")]:
    STAGE_DELTA_CONTRASTS.extend([
        {
            "contrast_name": "P2-Q_minus_P0",
            "left_condition": "P0",
            "right_condition": "P2-Q",
            "reranker_family": family,
            "reranker_role": role,
            "effect_label": "No-prior reranking effect",
        },
        {
            "contrast_name": "P2-P_minus_P2-Q",
            "left_condition": "P2-Q",
            "right_condition": "P2-P",
            "reranker_family": family,
            "reranker_role": role,
            "effect_label": "Stage 2 prior-feature effect",
        },
        {
            "contrast_name": "Full_minus_P2-P",
            "left_condition": "P2-P",
            "right_condition": "Full",
            "reranker_family": family,
            "reranker_role": role,
            "effect_label": "Personalized candidate-source effect",
        },
        {
            "contrast_name": "Full_minus_P2-Q",
            "left_condition": "P2-Q",
            "right_condition": "Full",
            "reranker_family": family,
            "reranker_role": role,
            "effect_label": "Combined personalization effect",
        },
        {
            "contrast_name": "b02_minus_P2-Q",
            "left_condition": "P2-Q",
            "right_condition": "b02",
            "reranker_family": family,
            "reranker_role": role,
            "effect_label": "Retrieval-personalization effect under no-prior reranking (b02 - a)",
        },
        {
            "contrast_name": "Full_minus_b02",
            "left_condition": "b02",
            "right_condition": "Full",
            "reranker_family": family,
            "reranker_role": role,
            "effect_label": "Prior-reranking effect under personalized retrieval (c - b02)",
        },
    ])

stage_delta_per_case_df = pd.concat(
    [build_primary_case_delta(contrast) for contrast in STAGE_DELTA_CONTRASTS],
    ignore_index=True,
    sort=False,
)
stage_delta_fold_lineage_qc = {
    "fold_id_available_in_notebook14_canonical_case_metrics": bool("fold_id" in stage_delta_per_case_df.columns),
    "lineage_evidence": "Notebook 14 canonical case metrics; upstream OOF fold IDs remain in Notebook 11/13 artifacts",
}
stage_delta_key = ["case_id", "contrast_name", "reranker_family"]
if stage_delta_per_case_df.duplicated(stage_delta_key).any():
    raise RuntimeError("Stage-delta per-case output contains duplicated keys.")

QCHS_APPLICABLE_CONTRASTS = {
    "P1-only_minus_P0", "Full_minus_P2-P", "Full_minus_P2-Q",
    "b02_minus_P2-Q", "Full_minus_b02"
}
summary_rows = []
for (contrast_name, family), group in stage_delta_per_case_df.groupby(
    ["contrast_name", "reranker_family"], sort=False, observed=True
):
    population_frames = [
        ("overall", group),
        ("Strong", group.loc[group["is_strong"]]),
        ("non-cold", group.loc[group["is_non_cold"]]),
    ]
    if contrast_name in QCHS_APPLICABLE_CONTRASTS:
        population_frames.append(("QCHS-active", group.loc[group["is_qchs_active"]]))
    for population, population_df in population_frames:
        if population_df.empty:
            continue
        first = population_df.iloc[0]
        stats = user_cluster_bootstrap_statistics(
            population_df,
            (CATEGORY_ID, contrast_name, family, population),
        )
        summary_rows.append({
            "category_id": CATEGORY_ID,
            "category_key": CATEGORY_KEY,
            "category_label": CATEGORY_LABEL,
            "contrast_name": contrast_name,
            "effect_label": first["effect_label"],
            "left_condition": first["left_condition"],
            "right_condition": first["right_condition"],
            "reranker_family": family,
            "reranker_role": first["reranker_role"],
            "candidate_pool_depth": REPORT_POOL_DEPTH,
            "metric_name": PRIMARY_METRIC_NAME,
            "metric_cutoff": PRIMARY_METRIC_CUTOFF,
            "population": population,
            **stats,
        })
stage_delta_paired_summary_df = pd.DataFrame(summary_rows)


def build_identity_qc(
    qc_population,
    left_condition,
    right_condition,
    family,
    subset_mode,
):
    left = primary_condition_rows(left_condition, family)
    right_all = primary_condition_rows(right_condition, family).copy()
    right_all["profile_fallback_flag"] = boolean_series(
        right_all["profile_fallback_flag"]
    ).fillna(False)
    regime = right_all["regime"].astype(str).str.lower()
    subset_excluded_case_count = 0
    if subset_mode == "cold":
        right = right_all.loc[regime.eq("cold")].copy()
        if right_condition in {"P1-only", "Full"}:
            subset_excluded_case_count = int(
                (~right["profile_fallback_flag"]).sum()
            )
    elif subset_mode == "qchs_fallback":
        right = right_all.loc[right_all["profile_fallback_flag"]].copy()
    elif subset_mode == "cold_qchs_fallback":
        cold_right = right_all.loc[regime.eq("cold")].copy()
        right = cold_right.loc[
            cold_right["profile_fallback_flag"]
        ].copy()
        subset_excluded_case_count = int(len(cold_right) - len(right))
    else:
        raise ValueError(f"Unsupported identity subset: {subset_mode}")

    expected_cases = set(right["case_id"].astype(str))
    left_cases = set(left["case_id"].astype(str))
    missing_left = expected_cases.difference(left_cases)
    paired = left.merge(
        right,
        on="case_id",
        how="inner",
        suffixes=("_left", "_right"),
        validate="one_to_one",
    )
    rank_left = pd.to_numeric(paired["target_rank_left"], errors="coerce")
    rank_right = pd.to_numeric(paired["target_rank_right"], errors="coerce")
    rank_equal = (
        (rank_left.isna() & rank_right.isna())
        | (rank_left.notna() & rank_right.notna() & rank_left.eq(rank_right))
    )
    metric_equal = np.isclose(
        pd.to_numeric(paired["metric_value_left"], errors="raise").to_numpy(dtype=float),
        pd.to_numeric(paired["metric_value_right"], errors="raise").to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-12,
    )
    query_equal = paired["query_id_left"].astype(str).eq(
        paired["query_id_right"].astype(str)
    )
    user_equal = paired["user_id_left"].astype(str).eq(
        paired["user_id_right"].astype(str)
    )
    regime_equal = paired["regime_left"].astype(str).eq(
        paired["regime_right"].astype(str)
    )
    expected_count = len(expected_cases)
    passed = bool(
        expected_count > 0
        and not missing_left
        and subset_excluded_case_count == 0
        and len(paired) == expected_count
        and rank_equal.all()
        and metric_equal.all()
        and query_equal.all()
        and user_equal.all()
        and regime_equal.all()
    )
    return {
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_label": CATEGORY_LABEL,
        "qc_population": qc_population,
        "subset_mode": subset_mode,
        "left_condition": left_condition,
        "right_condition": right_condition,
        "reranker_family": family,
        "candidate_pool_depth": REPORT_POOL_DEPTH,
        "metric_name": PRIMARY_METRIC_NAME,
        "metric_cutoff": PRIMARY_METRIC_CUTOFF,
        "expected_case_count": int(expected_count),
        "paired_case_count": int(len(paired)),
        "missing_left_case_count": int(len(missing_left)),
        "subset_excluded_case_count": int(subset_excluded_case_count),
        "target_rank_mismatch_count": int((~rank_equal).sum()),
        "ndcg_at_5_mismatch_count": int((~metric_equal).sum()),
        "query_id_mismatch_count": int((~query_equal).sum()),
        "user_id_mismatch_count": int((~user_equal).sum()),
        "regime_mismatch_count": int((~regime_equal).sum()),
        "candidate_level_identity_available": False,
        "identity_evidence": "canonical target_rank and reconstructed NDCG@5",
        "passed": passed,
    }


identity_qc_rows = [
    build_identity_qc(
        "cold Stage 1 fallback", "P0", "P1-only", "shared_stage1", "cold"
    ),
    build_identity_qc(
        "QCHS fallback Stage 1 identity",
        "P0", "P1-only", "shared_stage1", "qchs_fallback",
    ),
]
for family in PRIMARY_RERANKER_FAMILIES:
    identity_qc_rows.extend([
        build_identity_qc(
            "cold no-prior OOF fallback",
            "P2-Q", "P2-P", family, "cold",
        ),
        build_identity_qc(
            "cold combined fallback",
            "P2-Q", "Full", family, "cold_qchs_fallback",
        ),
        build_identity_qc(
            "QCHS fallback candidate-source identity",
            "P2-P", "Full", family, "qchs_fallback",
        ),
    ])
cold_fallback_identity_qc_df = pd.DataFrame(identity_qc_rows)

reconstructed_primary_metric = ndcg_from_rank(
    primary_metric_df["target_rank"], PRIMARY_METRIC_CUTOFF
)
canonical_metric_match = np.isclose(
    pd.to_numeric(primary_metric_df["metric_value"], errors="raise").to_numpy(dtype=float),
    reconstructed_primary_metric.to_numpy(dtype=float),
    rtol=0.0,
    atol=1e-12,
)

summary_key = [
    "reranker_method", "method_family", "stage_condition", "regime",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
]
direct_primary_summary = (
    primary_metric_df.groupby(summary_key, dropna=False, observed=True)
    .agg(
        direct_case_count=("case_id", "nunique"),
        direct_metric_mean=("metric_value", "mean"),
    )
    .reset_index()
)
saved_primary_summary = nb14_by_regime_pool_depth_df.loc[
    nb14_by_regime_pool_depth_df["candidate_pool_depth"].eq(REPORT_POOL_DEPTH)
    & nb14_by_regime_pool_depth_df["metric_name"].eq(PRIMARY_METRIC_NAME)
    & nb14_by_regime_pool_depth_df["metric_cutoff"].eq(PRIMARY_METRIC_CUTOFF)
][[*summary_key, "case_count", "metric_mean"]].copy()
summary_reconciliation = direct_primary_summary.merge(
    saved_primary_summary,
    on=summary_key,
    how="outer",
    indicator=True,
    validate="one_to_one",
)
summary_reconciliation["case_count_match"] = (
    summary_reconciliation["direct_case_count"].eq(summary_reconciliation["case_count"])
)
summary_reconciliation["metric_mean_match"] = np.isclose(
    pd.to_numeric(summary_reconciliation["direct_metric_mean"], errors="coerce"),
    pd.to_numeric(summary_reconciliation["metric_mean"], errors="coerce"),
    rtol=0.0,
    atol=1e-12,
    equal_nan=False,
)
summary_reconciliation_passed = bool(
    summary_reconciliation["_merge"].eq("both").all()
    and summary_reconciliation["case_count_match"].all()
    and summary_reconciliation["metric_mean_match"].all()
)

manifest_disagreement_path = (
    pipeline_manifest.get("source_metric_disagreement_path")
    or pipeline_manifest.get("output_paths", {}).get("source_metric_disagreements")
)
if manifest_disagreement_path:
    source_disagreement_path = Path(manifest_disagreement_path)
    source_disagreement_available = source_disagreement_path.exists()
else:
    source_disagreement_path = None
    source_disagreement_available = False

if source_disagreement_available:
    source_disagreement_df = pd.read_csv(source_disagreement_path)
    require_columns(
        source_disagreement_df,
        [
            "source_notebook", "condition", "reranker", "case_id",
            "candidate_pool_depth", "target_rank", "source_metric",
            "reconstructed_metric", "source_metric_name",
        ],
        "Notebook 14 source metric disagreements",
    )
else:
    source_disagreement_df = pd.DataFrame(columns=[
        "source_notebook", "condition", "reranker", "case_id",
        "candidate_pool_depth", "target_rank", "source_metric",
        "reconstructed_metric", "source_metric_name",
    ])

manifest_disagreement_count = pipeline_manifest.get("source_metric_disagreement_count")
if manifest_disagreement_count is None:
    manifest_disagreement_count = 0
source_disagreement_count_consistent = bool(
    (source_disagreement_available and len(source_disagreement_df) == int(manifest_disagreement_count))
    or (not source_disagreement_available and int(manifest_disagreement_count) == 0)
)
source_primary_disagreement_count = int(
    (
        pd.to_numeric(
            source_disagreement_df["candidate_pool_depth"], errors="coerce"
        ).eq(REPORT_POOL_DEPTH)
        & source_disagreement_df["source_metric_name"].astype(str).eq("NDCG@5")
    ).sum()
)

authority_rows = [
    {
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_label": CATEGORY_LABEL,
        "qc_check": "canonical NDCG@5 reconstruction",
        "source_notebook": NOTEBOOK_NAME,
        "stage_condition": "all canonical conditions",
        "reranker": "all available methods at depth 1000",
        "recorded_row_count": int(len(primary_metric_df)),
        "unique_recorded_case_count": int(primary_metric_df["case_id"].nunique()),
        "disagreement_count": int((~canonical_metric_match).sum()),
        "primary_contract_disagreement_count": int((~canonical_metric_match).sum()),
        "manifest_disagreement_count": int(manifest_disagreement_count),
        "artifact_available": True,
        "diagnostic_rows_are_capped_examples": False,
        "authority_source": "authoritative target_rank",
        "canonical_summary_uses_reconstructed_metric": True,
        "source_metric_agreement": bool(len(source_disagreement_df) == 0),
        "max_abs_metric_difference": float(np.max(np.abs(
            pd.to_numeric(primary_metric_df["metric_value"], errors="raise")
            .to_numpy(dtype=float)
            - reconstructed_primary_metric.to_numpy(dtype=float)
        ))),
        "record_semantics": "complete canonical rows at the primary contract",
        "check_passed": bool(canonical_metric_match.all()),
    },
    {
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_label": CATEGORY_LABEL,
        "qc_check": "Notebook 14 summary reconciliation",
        "source_notebook": pipeline_manifest.get("notebook_name", ""),
        "stage_condition": "all canonical conditions",
        "reranker": "all available methods at depth 1000",
        "recorded_row_count": int(len(summary_reconciliation)),
        "unique_recorded_case_count": int(primary_metric_df["case_id"].nunique()),
        "disagreement_count": int((~summary_reconciliation["metric_mean_match"]).sum()),
        "primary_contract_disagreement_count": int(
            (~summary_reconciliation["metric_mean_match"]).sum()
        ),
        "manifest_disagreement_count": int(manifest_disagreement_count),
        "artifact_available": True,
        "diagnostic_rows_are_capped_examples": False,
        "authority_source": "canonical per-case metric_value reconstructed from target_rank",
        "canonical_summary_uses_reconstructed_metric": True,
        "source_metric_agreement": bool(len(source_disagreement_df) == 0),
        "max_abs_metric_difference": float(np.nanmax(np.abs(
            pd.to_numeric(
                summary_reconciliation["direct_metric_mean"], errors="coerce"
            ).to_numpy(dtype=float)
            - pd.to_numeric(
                summary_reconciliation["metric_mean"], errors="coerce"
            ).to_numpy(dtype=float)
        ))),
        "record_semantics": "complete Notebook 14 summary cells at the primary contract",
        "check_passed": summary_reconciliation_passed,
    },
    {
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_label": CATEGORY_LABEL,
        "qc_check": "source metric disagreement inventory",
        "source_notebook": "all recorded sources",
        "stage_condition": "all recorded conditions",
        "reranker": "all recorded rerankers",
        "recorded_row_count": int(len(source_disagreement_df)),
        "unique_recorded_case_count": int(source_disagreement_df["case_id"].nunique()),
        "disagreement_count": int(len(source_disagreement_df)),
        "primary_contract_disagreement_count": source_primary_disagreement_count,
        "manifest_disagreement_count": int(manifest_disagreement_count),
        "artifact_available": bool(source_disagreement_available),
        "diagnostic_rows_are_capped_examples": True,
        "authority_source": "authoritative target_rank",
        "canonical_summary_uses_reconstructed_metric": True,
        "source_metric_agreement": bool(source_disagreement_df.empty),
        "max_abs_metric_difference": (
            float(np.max(np.abs(
                pd.to_numeric(
                    source_disagreement_df["source_metric"], errors="coerce"
                ).to_numpy(dtype=float)
                - pd.to_numeric(
                    source_disagreement_df["reconstructed_metric"], errors="coerce"
                ).to_numpy(dtype=float)
            )))
            if not source_disagreement_df.empty else 0.0
        ),
        "record_semantics": (
            "diagnostic examples; Notebook 14 records at most 10 rows "
            "per source metric column"
        ),
        "check_passed": source_disagreement_count_consistent,
    },
]
if not source_disagreement_df.empty:
    for (source_notebook, condition, reranker), group in source_disagreement_df.groupby(
        ["source_notebook", "condition", "reranker"],
        dropna=False,
        observed=True,
    ):
        authority_rows.append({
            "category_id": CATEGORY_ID,
            "category_key": CATEGORY_KEY,
            "category_label": CATEGORY_LABEL,
            "qc_check": "source metric disagreement by run",
            "source_notebook": str(source_notebook),
            "stage_condition": str(condition),
            "reranker": str(reranker),
            "recorded_row_count": int(len(group)),
            "unique_recorded_case_count": int(group["case_id"].nunique()),
            "disagreement_count": int(len(group)),
            "primary_contract_disagreement_count": int(
                (
                    pd.to_numeric(
                        group["candidate_pool_depth"], errors="coerce"
                    ).eq(REPORT_POOL_DEPTH)
                    & group["source_metric_name"].astype(str).eq("NDCG@5")
                ).sum()
            ),
            "manifest_disagreement_count": int(manifest_disagreement_count),
            "artifact_available": True,
            "diagnostic_rows_are_capped_examples": True,
            "authority_source": "authoritative target_rank",
            "canonical_summary_uses_reconstructed_metric": True,
            "source_metric_agreement": False,
            "max_abs_metric_difference": float(np.max(np.abs(
                pd.to_numeric(group["source_metric"], errors="raise")
                .to_numpy(dtype=float)
                - pd.to_numeric(group["reconstructed_metric"], errors="raise")
                .to_numpy(dtype=float)
            ))),
            "record_semantics": (
                "diagnostic examples; Notebook 14 records at most 10 rows "
                "per source metric column"
            ),
            "check_passed": True,
        })
aggregate_metric_authority_qc_df = pd.DataFrame(authority_rows)


In [20]:
# ==== Final Descriptive Validation, Exports, and Manifest ====
all_outputs = [
    stage_allocation_overall_by_reranker_df,
    stage_allocation_by_regime_reranker_df,
    stage_allocation_by_regime_pool_depth_df,
    stage_allocation_by_pool_depth_df,
    stage_allocation_five_condition_comparison_df,
    stage_allocation_paired_contrasts_df,
    stage_allocation_significance_tests_df,
    stage_allocation_qchs_coverage_df,
    stage_allocation_qchs_active_subset_df,
    stage_allocation_method_availability_df,
    stage_delta_per_case_df,
    stage_delta_paired_summary_df,
    cold_fallback_identity_qc_df,
    aggregate_metric_authority_qc_df,
]
for index, output_df in enumerate(all_outputs):
    require_unique_columns(output_df, f"Notebook 15 output {index}")

availability_statuses = set(
    stage_allocation_method_availability_df["availability_status"].astype(str)
)
allowed_statuses = {
    "available", "not_run_by_design",
    "missing_unexpected", "invalid_cutoff_for_pool",
}
if not availability_statuses.issubset(allowed_statuses):
    raise RuntimeError(f"Unexpected availability statuses: {sorted(availability_statuses)}")

main_full_contrasts = stage_allocation_paired_contrasts_df.loc[
    stage_allocation_paired_contrasts_df["contrast_name"].eq("Full_minus_P2-P")
    & stage_allocation_paired_contrasts_df["user_scope"].eq("overall")
]
observed_full_minus_p2p_families = set(main_full_contrasts["reranker_family"].astype(str))
expected_full_minus_p2p_families = set(RERANKER_FAMILIES)
if observed_full_minus_p2p_families != expected_full_minus_p2p_families:
    raise RuntimeError(
        "Full minus P2-P does not retain every reranker family: "
        f"missing={sorted(expected_full_minus_p2p_families - observed_full_minus_p2p_families)}, "
        f"extra={sorted(observed_full_minus_p2p_families - expected_full_minus_p2p_families)}. "
        "Rerun the helper and paired-contrast cells before final validation."
    )

expected_fallback = int(
    pipeline_manifest.get("qchs_fallback_preservation", {}).get(
        "observed_fallback_case_count", 0
    ) or 0
)
observed_fallback = int(
    p1_case_contract_df.loc[
        p1_case_contract_df["profile_fallback_flag"].fillna(False), "case_id"
    ].nunique()
)
if observed_fallback != expected_fallback:
    raise RuntimeError(
        f"QCHS fallback count disagrees with Notebook 14: "
        f"expected={expected_fallback}, observed={observed_fallback}"
    )

stage_allocation_overall_by_reranker_df.to_csv(
    OUTPUT_FILES["overall_by_reranker"], index=False, encoding="utf-8-sig"
)
stage_allocation_by_regime_reranker_df.to_csv(
    OUTPUT_FILES["by_regime_reranker"], index=False, encoding="utf-8-sig"
)
stage_allocation_by_regime_pool_depth_df.to_csv(
    OUTPUT_FILES["by_regime_pool_depth"], index=False, encoding="utf-8-sig"
)
stage_allocation_by_pool_depth_df.to_csv(
    OUTPUT_FILES["by_pool_depth"], index=False, encoding="utf-8-sig"
)
stage_allocation_five_condition_comparison_df.to_csv(
    OUTPUT_FILES["five_condition_comparison"], index=False, encoding="utf-8-sig"
)
stage_allocation_paired_contrasts_df.to_csv(
    OUTPUT_FILES["paired_contrasts"], index=False, encoding="utf-8-sig"
)
stage_allocation_significance_tests_df.to_csv(
    OUTPUT_FILES["significance_tests"], index=False, encoding="utf-8-sig"
)
stage_allocation_qchs_coverage_df.to_csv(
    OUTPUT_FILES["qchs_coverage"], index=False, encoding="utf-8-sig"
)
stage_allocation_qchs_active_subset_df.to_csv(
    OUTPUT_FILES["qchs_active_subset"], index=False, encoding="utf-8-sig"
)
stage_allocation_method_availability_df.to_csv(
    OUTPUT_FILES["method_availability"], index=False, encoding="utf-8-sig"
)
stage_delta_per_case_df.to_parquet(
    OUTPUT_FILES["stage_delta_per_case"], index=False
)
stage_delta_paired_summary_df.to_csv(
    OUTPUT_FILES["stage_delta_paired_summary"], index=False, encoding="utf-8-sig"
)
cold_fallback_identity_qc_df.to_csv(
    OUTPUT_FILES["cold_fallback_identity_qc"], index=False, encoding="utf-8-sig"
)
aggregate_metric_authority_qc_df.to_csv(
    OUTPUT_FILES["aggregate_metric_authority_qc"], index=False, encoding="utf-8-sig"
)

required_primary_contrasts = {
    "P1-only_minus_P0", "P2-Q_minus_P0", "P2-P_minus_P2-Q",
    "Full_minus_P2-P", "Full_minus_P2-Q",
    "b02_minus_P2-Q", "Full_minus_b02",
}
observed_primary_contrasts = set(stage_delta_paired_summary_df["contrast_name"])
if observed_primary_contrasts != required_primary_contrasts:
    raise RuntimeError(
        f"Primary stage-delta contrast mismatch: {sorted(observed_primary_contrasts)}"
    )
required_populations = {"overall", "Strong", "non-cold"}
for contrast_name in required_primary_contrasts:
    observed_populations = set(
        stage_delta_paired_summary_df.loc[
            stage_delta_paired_summary_df["contrast_name"].eq(contrast_name),
            "population",
        ]
    )
    if not required_populations.issubset(observed_populations):
        raise RuntimeError(
            f"Missing primary populations for {contrast_name}: "
            f"{sorted(required_populations.difference(observed_populations))}"
        )

canonical_metric_reconstruction_passed = bool(canonical_metric_match.all())
blocking_identity_qc_df = cold_fallback_identity_qc_df.loc[
    cold_fallback_identity_qc_df["qc_population"].isin([
        "cold Stage 1 fallback",
        "QCHS fallback Stage 1 identity",
        "cold no-prior OOF fallback",
    ])
].copy()

identity_qc_all_passed = bool(blocking_identity_qc_df["passed"].all())

diagnostic_identity_qc_all_passed = bool(
    cold_fallback_identity_qc_df["passed"].all()
)

ready_for_thesis_reporting = bool(
    canonical_metric_reconstruction_passed
    and summary_reconciliation_passed
    and source_disagreement_count_consistent
    and identity_qc_all_passed
)

manifest = {
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_label": CATEGORY_LABEL,
    "analysis_output_dir": str(OUT_DIR),
    "notebook14_manifest_path": str(PIPELINE_MANIFEST_PATH),
    "notebook14_canonical_raw_path": str(CANONICAL_RAW_PATH),
    "notebook14_canonical_primary_path": str(CANONICAL_PRIMARY_PATH),
    "notebook14_canonical_raw_sha256": file_sha256(CANONICAL_RAW_PATH),
    "notebook14_canonical_primary_sha256": file_sha256(CANONICAL_PRIMARY_PATH),
    "notebook14_ready_for_downstream": bool(
        pipeline_manifest.get("ready_for_downstream") is True
    ),
    "canonical_conditions": CANONICAL_CONDITIONS,
    "reranker_families": RERANKER_FAMILIES,
    "candidate_pool_depths": pipeline_manifest["pool_depth_contract"][
        "candidate_pool_depths"
    ],
    "metric_cutoffs": pipeline_manifest["pool_depth_contract"]["metric_cutoffs"],
    "metric_names": METRIC_NAMES,
    "contrast_definitions": {
        "P1-only_minus_P0": "shared Stage 1 retrieval-personalization effect",
        "P2-Q_minus_P0": "reranking effect without user-prior features",
        "P2-P_minus_P2-Q": "Stage 2 prior-feature effect",
        "Full_minus_P2-P": "personalized candidate-source effect",
        "Full_minus_P2-Q": "combined personalization effect",
    },
    "primary_evaluation_contract": {
        "candidate_pool_depth": REPORT_POOL_DEPTH,
        "metric_name": PRIMARY_METRIC_NAME,
        "metric_cutoff": PRIMARY_METRIC_CUTOFF,
        "primary_reranker": "lightgbm",
        "secondary_reranker": "transformer",
    },
    "analysis_role": "descriptive_stage_allocation_only",
    "confirmatory_inference_authority": False,
    "confirmatory_inference_notebook": "16_paired_significance_summary",
    "legacy_significance_output_policy": "p_values_not_computed_notebook16_is_inference_authority",
    "gam_depth_policy": "single_K1000_fit_evaluated_on_fixed_prefixes_100_300_500_700_1000",
    "resampling_contract": {
        "role": "descriptive_schema_only_p_values_not_computed",
        "paired_bootstrap_resamples": N_BOOTSTRAP,
        "paired_bootstrap_unit": "user_id cluster",
        "sign_flip_permutations": N_SIGN_FLIP,
        "random_seed_root": RANDOM_SEED,
        "multiplicity_correction": "none_descriptive_only",
        "alpha": ALPHA,
    },
    "shared_stage1_policy": {
        "inferential_rows_repeated": False,
        "presentation_rows_repeated": True,
        "P1-only_minus_P0_test_counted_once": True,
    },
    "report_pool_depth": REPORT_POOL_DEPTH,
    "report_depth_status": "fixed_primary_contract",
    "metric_authority": {
        "authoritative_field": "target_rank",
        "canonical_ndcg_at_5_reconstructed": canonical_metric_reconstruction_passed,
        "notebook14_summary_reconciled": summary_reconciliation_passed,
        "source_metric_disagreement_artifact_consistent": (
            source_disagreement_count_consistent
        ),
        "recorded_source_metric_disagreement_count": int(
            len(source_disagreement_df)
        ),
        "diagnostic_rows_are_capped_examples": True,
    },
    "qchs_fallback_cases_included_in_main": True,
    "qchs_active_subset_is_diagnostic": True,
    "stage_delta_fold_lineage_qc": stage_delta_fold_lineage_qc,
    "output_files": {name: str(path) for name, path in OUTPUT_FILES.items()},
    "validation_results": {
        "notebook14_hashes_verified": notebook14_hashes_verified,
        "notebook14_primary_grid_complete": notebook14_primary_grid_complete,
        "notebook14_full_transfer_readiness": bool(
            pipeline_manifest.get("full_transfer_readiness") is True
        ),
        "contrasts_derived_only_from_notebook14": True,
        "canonical_key_unique": True,
        "all_reranker_families_retained": True,
        "gam_depth_1000_only": True,
        "shared_stage1_contrast_not_repeated": True,
        "paired_query_universes_equal": True,
        "qchs_fallback_cases_preserved": True,
        "case_query_user_grain_valid": True,
        "stage_delta_fold_id_available_in_notebook14": stage_delta_fold_lineage_qc["fold_id_available_in_notebook14_canonical_case_metrics"],
        "canonical_ndcg_at_5_matches_target_rank": (
            canonical_metric_reconstruction_passed
        ),
        "notebook14_summary_matches_canonical_raw": (
            summary_reconciliation_passed
        ),
        "source_disagreement_count_consistent": (
            source_disagreement_count_consistent
        ),
          "blocking_fallback_identity_passed": identity_qc_all_passed,
        "diagnostic_fallback_identity_passed": diagnostic_identity_qc_all_passed,
        "required_primary_contrasts_present": True,
        "required_primary_populations_present": True,
        "output_columns_unique": True,
    },
    "ready_for_thesis_reporting": ready_for_thesis_reporting,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
OUTPUT_FILES["manifest"].write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

failed_identity_qc = cold_fallback_identity_qc_df.loc[
    ~cold_fallback_identity_qc_df["passed"]
]
if not failed_identity_qc.empty:
    print("WARNING: Cold or QCHS fallback identity QC failed.")
    display(failed_identity_qc)

print("Stage allocation outputs written to:", OUT_DIR)
print("Primary report contract: NDCG@5 at candidate pool depth 1000.")
print("Ready for thesis reporting:", ready_for_thesis_reporting)
if not ready_for_thesis_reporting:
    raise RuntimeError(
        "Notebook 15 validation failed; descriptive outputs are not ready for reporting."
    )


,category_id,category_key,category_label,qc_population,subset_mode,left_condition,right_condition,reranker_family,candidate_pool_depth,metric_name,...,missing_left_case_count,subset_excluded_case_count,target_rank_mismatch_count,ndcg_at_5_mismatch_count,query_id_mismatch_count,user_id_mismatch_count,regime_mismatch_count,candidate_level_identity_available,identity_evidence,passed
4,herbal,herbal_supplements,Herbal Supplements,QCHS fallback candidate-source identity,qchs_fallback,P2-P,Full,lightgbm,1000,NDCG,...,0,0,341,81,0,0,0,False,canonical target_rank and reconstructed NDCG@5,False
6,herbal,herbal_supplements,Herbal Supplements,cold combined fallback,cold_qchs_fallback,P2-Q,Full,transformer,1000,NDCG,...,0,0,33,1,0,0,0,False,canonical target_rank and reconstructed NDCG@5,False
7,herbal,herbal_supplements,Herbal Supplements,QCHS fallback candidate-source identity,qchs_fallback,P2-P,Full,transformer,1000,NDCG,...,0,0,392,87,0,0,0,False,canonical target_rank and reconstructed NDCG@5,False


Stage allocation outputs written to: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/stage_allocation_summary
Primary report contract: NDCG@5 at candidate pool depth 1000.
Ready for thesis reporting: True


In [21]:
# ==== Final Verification Report Export ====
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math

try:
    import numpy as _report_np
except Exception:
    _report_np = None
try:
    import pandas as _report_pd
except Exception:
    _report_pd = globals().get("pd")


def _report_is_dataframe(value):
    return _report_pd is not None and isinstance(value, _report_pd.DataFrame)


def _report_is_missing(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if _report_pd is not None:
        try:
            missing = _report_pd.isna(value)
            if isinstance(missing, (bool, type(None))):
                return bool(missing)
        except Exception:
            pass
    return False


def _report_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if _report_np is not None and isinstance(value, _report_np.generic):
        return _report_jsonable(value.item())
    if _report_is_missing(value):
        return None
    if isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return None if math.isnan(value) else value
    if isinstance(value, dict):
        return {str(k): _report_jsonable(v) for k, v in value.items()}
    if _report_pd is not None:
        if isinstance(value, _report_pd.Series):
            return [_report_jsonable(v) for v in value.tolist()]
        if _report_is_dataframe(value):
            return _report_frame_records(value)
    if isinstance(value, (list, tuple, set)):
        return [_report_jsonable(v) for v in value]
    if hasattr(value, "tolist"):
        try:
            return _report_jsonable(value.tolist())
        except Exception:
            pass
    if hasattr(value, "item"):
        try:
            return _report_jsonable(value.item())
        except Exception:
            pass
    return str(value)


def _report_frame_records(frame, limit=50, columns=None, drop_case_columns=True):
    if not _report_is_dataframe(frame):
        return []
    work = frame.copy()
    if columns is not None:
        keep = [column for column in columns if column in work.columns]
        work = work[keep]
    if drop_case_columns:
        disallowed = {
            "case_id", "query_id", "user_id", "item_id", "parent_asin",
            "target_parent_asin", "target_item_id", "review_id",
        }
        drop = [column for column in work.columns if str(column).lower() in disallowed]
        if drop:
            work = work.drop(columns=drop)
    records = [_report_jsonable(row) for row in work.head(limit).to_dict(orient="records")]
    if len(work) > limit:
        records.append({"note": "truncated", "row_count": int(len(work)), "rows_emitted": int(limit)})
    return records


def _report_output_dir():
    if "OUT_DIR" in globals():
        return Path(globals()["OUT_DIR"])
    if "OUTPUT_DIR" in globals():
        return Path(globals()["OUTPUT_DIR"])
    raise RuntimeError("No existing output-directory variable was found; expected OUT_DIR or OUTPUT_DIR.")


_report_dir = _report_output_dir() / "report"
_report_dir.mkdir(parents=True, exist_ok=True)


def _report_is_inside(path, parent):
    try:
        Path(path).resolve().relative_to(Path(parent).resolve())
        return True
    except Exception:
        return False


def _report_file_sha256(path, allow_heavy=False):
    try:
        path = Path(path)
    except Exception:
        return None
    if not path.exists() or not path.is_file():
        return None
    if not allow_heavy and path.suffix.lower() in {".parquet", ".pkl", ".pickle", ".joblib", ".pt", ".pth", ".bin"}:
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _report_notebook_path():
    candidates = []
    for key in ["__vsc_ipynb_file__", "__file__"]:
        value = globals().get(key)
        if value:
            candidates.append(Path(value))
    notebook_name = globals().get("NOTEBOOK_NAME")
    if notebook_name:
        candidates.append(Path.cwd() / str(notebook_name))
        project_root = globals().get("PROJECT_ROOT")
        if project_root:
            candidates.append(Path(project_root) / str(notebook_name))
    for candidate in candidates:
        try:
            if candidate.exists() and candidate.suffix.lower() == ".ipynb":
                return candidate
        except Exception:
            pass
    return None


_report_nb_path = _report_notebook_path()
_report_notebook = _report_nb_path.name if _report_nb_path is not None else str(globals().get("NOTEBOOK_NAME", "unknown_notebook"))
_report_category = globals().get("CATEGORY_ID", globals().get("CATEGORY_LABEL", "cross_category"))


def _report_path_from_maps(key):
    for map_name in ["OUTPUT_FILES", "OUTPUT_PATHS", "output_paths"]:
        mapping = globals().get(map_name)
        if isinstance(mapping, dict) and key in mapping:
            return str(mapping[key])
    return None


def _report_path_from_var(name):
    value = globals().get(name)
    return str(value) if value is not None else None


def _report_row_value(row, candidates):
    for column in candidates:
        if column in row and not _report_is_missing(row[column]):
            return row[column]
    return None


def _report_ci(row, low_candidates, high_candidates):
    lo = _report_row_value(row, low_candidates)
    hi = _report_row_value(row, high_candidates)
    if _report_is_missing(lo) or _report_is_missing(hi):
        return None
    return [_report_jsonable(lo), _report_jsonable(hi)]


def _report_value(claim_id, value=None, ci=None, p=None, n=None, source_file=None, aggregation=None, note=None):
    record = {
        "claim_id": str(claim_id),
        "value": _report_jsonable(value),
        "ci": _report_jsonable(ci),
        "p": _report_jsonable(p),
        "n": _report_jsonable(n),
        "source_file": _report_jsonable(source_file),
        "aggregation": _report_jsonable(aggregation),
    }
    if note:
        record["note"] = str(note)
    return record


def _report_missing_value(claim_id, note="not_available_in_notebook"):
    return _report_value(
        claim_id=claim_id,
        value=None,
        ci=None,
        p=None,
        n=None,
        source_file=None,
        aggregation="not_available_in_notebook",
        note=note,
    )


def _report_gate(gate_id, observed=None, expected_contract="", self_flag=False, note=None):
    record = {
        "gate_id": str(gate_id),
        "observed": _report_jsonable(observed),
        "expected_contract": str(expected_contract),
        "self_flag": bool(self_flag),
    }
    if note:
        record["note"] = str(note)
    return record


def _report_frame_failed(frame, passed_columns=("passed", "check_passed", "identity_passed"), status_columns=("status", "coverage_status")):
    if not _report_is_dataframe(frame) or frame.empty:
        return False
    for column in passed_columns:
        if column in frame.columns:
            try:
                return not bool(frame[column].astype(bool).all())
            except Exception:
                pass
    for column in status_columns:
        if column in frame.columns:
            statuses = frame[column].astype(str).str.upper()
            return not bool(statuses.isin(["PASS", "SUCCESS", "TRUE"]).all())
    return False


def _report_filter_primary(frame):
    work = frame.copy()
    original = work
    if "metric_name" in work.columns:
        filtered = work.loc[work["metric_name"].astype(str).eq(str(globals().get("PRIMARY_METRIC_NAME", "NDCG")))]
        if not filtered.empty:
            work = filtered
    if "metric_cutoff" in work.columns:
        filtered = work.loc[_report_pd.to_numeric(work["metric_cutoff"], errors="coerce").eq(int(globals().get("PRIMARY_METRIC_CUTOFF", 5)))]
        if not filtered.empty:
            work = filtered
    if "candidate_pool_depth" in work.columns and "REPORT_POOL_DEPTH" in globals():
        filtered = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").eq(int(globals()["REPORT_POOL_DEPTH"]))]
        if not filtered.empty:
            work = filtered
    return work if not work.empty else original


def _report_values_15():
    values = []
    five = globals().get("stage_allocation_five_condition_comparison_df")
    if _report_is_dataframe(five):
        work = _report_filter_primary(five)
        value_columns = ["mean_metric_value", "metric_mean", "mean_value", "mean", "metric_value_mean", "mean_ndcg_at_5"]
        value_column = next((column for column in value_columns if column in work.columns), None)
        if value_column is None:
            values.append(_report_missing_value("section6_five_condition_means", "value_column_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                parts = [
                    row.get("stage_condition"), row.get("reranker_family"),
                    row.get("candidate_pool_depth"), row.get("metric_name"), row.get("metric_cutoff"),
                ]
                claim_id = "section6_condition_mean:" + "|".join(str(part) for part in parts if not _report_is_missing(part))
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get(value_column),
                    ci=_report_ci(row, ["bootstrap_ci_95_low", "ci_low", "lower_ci"], ["bootstrap_ci_95_high", "ci_high", "upper_ci"]),
                    p=_report_row_value(row, ["p", "p_value", "sign_flip_p_value"]),
                    n=_report_row_value(row, ["case_count", "query_count", "n_cases", "n", "sample_size"]),
                    source_file=_report_path_from_maps("five_condition_comparison"),
                    aggregation=f"{value_column} from stage_allocation_five_condition_comparison_df",
                ))
    else:
        values.append(_report_missing_value("section6_five_condition_means"))

    contrasts = globals().get("stage_allocation_paired_contrasts_df")
    if _report_is_dataframe(contrasts):
        work = contrasts.copy()
        if "contrast_name" in work.columns:
            work = work.loc[work["contrast_name"].astype(str).eq("P2-P_minus_P2-Q")]
        if "metric_name" in work.columns:
            work = work.loc[work["metric_name"].astype(str).eq("NDCG")]
        if "metric_cutoff" in work.columns:
            work = work.loc[_report_pd.to_numeric(work["metric_cutoff"], errors="coerce").eq(5)]
        if "candidate_pool_depth" in work.columns:
            work = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").isin([100, 300, 500, 700, 1000])]
        if "user_scope" in work.columns:
            work = work.loc[work["user_scope"].astype(str).eq("overall")]
        if "analysis_subset" in work.columns:
            work = work.loc[work["analysis_subset"].astype(str).eq("all_cases")]
        if work.empty:
            values.append(_report_missing_value("fig6_3_s2p_minus_s2q_ndcg5_by_depth", "requested_contrast_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                claim_id = "fig6_3_s2p_minus_s2q_ndcg5_by_depth:" + "|".join(str(row.get(column)) for column in ["reranker_family", "candidate_pool_depth"] if column in row)
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get("mean_difference"),
                    ci=_report_ci(row, ["bootstrap_ci_95_low", "ci_low"], ["bootstrap_ci_95_high", "ci_high"]),
                    p=_report_row_value(row, ["sign_flip_p_value", "p_value", "p"]),
                    n=_report_row_value(row, ["paired_query_count", "n_pairs", "case_count", "n"]),
                    source_file=_report_path_from_maps("paired_contrasts"),
                    aggregation="existing paired mean_difference for P2-P_minus_P2-Q by depth",
                ))
    else:
        values.append(_report_missing_value("fig6_3_s2p_minus_s2q_ndcg5_by_depth"))
    return values


def _report_gates_15():
    gates = []
    manifest_obj = globals().get("manifest", {}) if isinstance(globals().get("manifest", {}), dict) else {}
    sig = globals().get("stage_allocation_significance_tests_df")
    observed_scope = {
        "analysis_role": manifest_obj.get("analysis_role"),
        "confirmatory_inference_authority_values": sorted(sig["confirmatory_inference_authority"].dropna().astype(str).unique().tolist()) if _report_is_dataframe(sig) and "confirmatory_inference_authority" in sig.columns else None,
        "inference_role_values": sorted(sig["inference_role"].dropna().astype(str).unique().tolist()) if _report_is_dataframe(sig) and "inference_role" in sig.columns else None,
    }
    gates.append(_report_gate("descriptive_only_scope", observed_scope, "descriptive-only; confirmatory inference authority remains Notebook 16", False))
    authority = globals().get("aggregate_metric_authority_qc_df")
    gates.append(_report_gate(
        "aggregate_metric_authority_qc",
        _report_frame_records(authority, limit=30),
        "descriptive aggregates reconcile with canonical authority",
        _report_frame_failed(authority, passed_columns=("check_passed", "passed")),
        None if _report_is_dataframe(authority) else "not_available_in_notebook",
    ))
    identity = globals().get("cold_fallback_identity_qc_df")
    gates.append(_report_gate(
        "cold_fallback_identity_qc",
        _report_frame_records(identity, limit=30),
        "fallback identity diagnostics are descriptive QC only",
        _report_frame_failed(identity),
        None if _report_is_dataframe(identity) else "not_available_in_notebook",
    ))
    gates.append(_report_gate(
        "stage_delta_fold_lineage_qc",
        manifest_obj.get("stage_delta_fold_lineage_qc", globals().get("stage_delta_fold_lineage_qc")),
        "stage-delta lineage recorded from existing Notebook 14/15 artifacts",
        False,
        None if (manifest_obj.get("stage_delta_fold_lineage_qc") is not None or "stage_delta_fold_lineage_qc" in globals()) else "not_available_in_notebook",
    ))
    return gates


def _report_values_18():
    values = []
    frame = globals().get("primary_feature_group_gain_summary_df")
    source_key = "primary_feature_group_gain_summary"
    if not _report_is_dataframe(frame):
        frame = globals().get("feature_group_gain_summary_df")
        source_key = "feature_group_gain_summary"
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "primary_report_depth" in work.columns:
            filtered = work.loc[work["primary_report_depth"].astype(bool)]
            if not filtered.empty:
                work = filtered
        elif "candidate_pool_depth" in work.columns and "PRIMARY_POOL_DEPTH" in globals():
            filtered = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").eq(int(globals()["PRIMARY_POOL_DEPTH"]))]
            if not filtered.empty:
                work = filtered
        if "feature_group" not in work.columns or "mean_normalized_gain" not in work.columns:
            values.append(_report_missing_value("fig7_1_grouped_normalized_gain_pct", "required_columns_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                values.append(_report_value(
                    claim_id=f"fig7_1_grouped_normalized_gain_pct:{row.get('feature_group')}",
                    value=None if _report_is_missing(row.get("mean_normalized_gain")) else float(row.get("mean_normalized_gain")) * 100.0,
                    ci=None,
                    p=None,
                    n=_report_row_value(row, ["fold_count", "n_folds", "n"]),
                    source_file=_report_path_from_maps(source_key),
                    aggregation="mean_normalized_gain across folds, expressed as percent",
                ))
    else:
        values.append(_report_missing_value("fig7_1_grouped_normalized_gain_pct"))
    return values


def _report_gates_18():
    gates = []
    run = globals().get("run_manifest", {}) if isinstance(globals().get("run_manifest", {}), dict) else {}
    share_observed = {
        "manifest_flag": run.get("grouped_gain_shares_sum_to_one"),
        "primary_share_total": globals().get("_primary_share_total"),
        "group_share_totals": globals().get("_group_share_totals"),
    }
    share_self_flag = False
    if share_observed["manifest_flag"] is not None:
        share_self_flag = share_observed["manifest_flag"] is not True
    gates.append(_report_gate("grouped_gain_shares_sum_to_one", share_observed, "grouped gain shares sum to one within existing fold/depth summaries", share_self_flag))
    perm_flag = run.get("permutation_importance_computed")
    gates.append(_report_gate("permutation_importance_computed", perm_flag, "false in slim build manifest", perm_flag is not False, None if perm_flag is not None else "not_available_in_notebook"))
    shap_flag = run.get("shap_computed")
    gates.append(_report_gate("shap_computed", shap_flag, "false in slim build manifest", shap_flag is not False, None if shap_flag is not None else "not_available_in_notebook"))
    reproduction = globals().get("reproduction_qc_df")
    gates.append(_report_gate(
        "native_fold_prediction_reproduction",
        _report_frame_records(reproduction, limit=30),
        "native fold predictions reproduce strict OOF exports within recorded tolerance",
        _report_frame_failed(reproduction, passed_columns=("prediction_reproduced", "passed")),
        None if _report_is_dataframe(reproduction) else "not_available_in_notebook",
    ))
    return gates


def _report_values_19():
    values = []
    frame = globals().get("branch_contribution_df")
    required = list(globals().get("REQUIRED_BRANCH_GROUPS", ["candidate_common", "functional_prior", "brand_prior"]))
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "branch_group" in work.columns:
            filtered = work.loc[work["branch_group"].astype(str).isin(required)]
            if not filtered.empty:
                work = filtered
        if "importance_mean" not in work.columns:
            values.append(_report_missing_value("table7_1_branch_masking_delta", "importance_mean_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                parts = [row.get("branch_group"), row.get("user_scope"), row.get("regime")]
                claim_id = "table7_1_branch_masking_delta:" + "|".join(str(part) for part in parts if not _report_is_missing(part))
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get("importance_mean"),
                    ci=_report_ci(row, ["importance_ci_95_low", "ci_low"], ["importance_ci_95_high", "ci_high"]),
                    p=None,
                    n=_report_row_value(row, ["case_count", "fold_count", "n"]),
                    source_file=_report_path_from_maps("branch_contribution"),
                    aggregation="mean NDCG@5 decrease from existing branch masking summary",
                ))
    else:
        values.append(_report_missing_value("table7_1_branch_masking_delta"))
    return values


def _report_gates_19():
    gates = []
    manifest_obj = globals().get("analysis_manifest", {}) if isinstance(globals().get("analysis_manifest", {}), dict) else {}
    checkpoint = globals().get("checkpoint_diagnostics_df")
    checkpoint_values = checkpoint["checkpoint_selection_metric"].dropna().astype(str).unique().tolist() if _report_is_dataframe(checkpoint) and "checkpoint_selection_metric" in checkpoint.columns else None
    observed_checkpoint = {
        "manifest_checkpoint_selection_metric": manifest_obj.get("checkpoint_selection_metric"),
        "checkpoint_diagnostics_values": checkpoint_values,
    }
    checkpoint_bad = manifest_obj.get("checkpoint_selection_metric") not in (None, "validation_ndcg_at_5")
    if checkpoint_values is not None:
        checkpoint_bad = checkpoint_bad or any(value != "validation_ndcg_at_5" for value in checkpoint_values)
    gates.append(_report_gate("checkpoint_selection_metric", observed_checkpoint, "validation_ndcg_at_5", checkpoint_bad))
    manifests = globals().get("interpretation_manifests", {}) if isinstance(globals().get("interpretation_manifests", {}), dict) else {}
    observed_mismatch = {
        "analysis_manifest_disabled_category_mismatch": manifest_obj.get("disabled_category_mismatch"),
        "full_manifest_disabled_category_mismatch": manifests.get("Full", {}).get("disabled_category_mismatch") if isinstance(manifests.get("Full", {}), dict) else None,
    }
    mismatch_bad = any(value is not None and value is not False for value in observed_mismatch.values())
    gates.append(_report_gate("disabled_category_mismatch", observed_mismatch, "false", mismatch_bad))
    return gates


def _report_values_20():
    values = []
    frame = globals().get("scorecard")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            claim_id = "table3_e1_structural_contrast:" + "|".join(str(row.get(column)) for column in ["claim_dimension", "metric"] if column in row)
            values.append(_report_value(
                claim_id=claim_id,
                value=row.get("value"),
                ci=None,
                p=None,
                n=_report_row_value(row, ["n", "case_count", "item_count", "query_count"]),
                source_file=str(_report_output_dir() / "schema_audit_thesis_evidence_scorecard.csv"),
                aggregation="existing schema-audit scorecard value",
            ))
    else:
        values.append(_report_missing_value("table3_e1_structural_contrast_values"))
    return values


def _report_gates_20():
    frame = globals().get("qc_summary")
    return [_report_gate(
        "schema_audit_qc_summary",
        _report_frame_records(frame, limit=50),
        "schema-audit QC rows reported by notebook",
        _report_frame_failed(frame, status_columns=("status",)),
        None if _report_is_dataframe(frame) else "not_available_in_notebook",
    )]


def _report_values_22():
    values = []
    frame = globals().get("paired_summary")
    requested = {
        "Actual_minus_Shuffled",
        "Actual_minus_QCHSfiltered",
        "Actual_minus_QCHS_filtered",
        "QCHS_minus_Actual",
        "Actual_minus_No_User_Brand",
    }
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "population" in work.columns:
            work = work.loc[work["population"].astype(str).eq("non-cold")]
        if "contrast" in work.columns:
            work = work.loc[work["contrast"].astype(str).isin(requested)]
        if work.empty:
            values.append(_report_missing_value("table_a_2_prior_policy_non_cold_deltas", "requested_non_cold_contrasts_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                values.append(_report_value(
                    claim_id=f"table_a_2_prior_policy_delta:{row.get('contrast')}|non-cold",
                    value=row.get("mean_delta"),
                    ci=_report_ci(row, ["ci_low", "bootstrap_ci_95_low"], ["ci_high", "bootstrap_ci_95_high"]),
                    p=_report_row_value(row, ["p", "p_value"]),
                    n=_report_row_value(row, ["n_cases", "n", "case_count"]),
                    source_file=_report_path_from_var("PAIRED_SUMMARY_PATH"),
                    aggregation="existing non-cold paired_summary mean_delta and CI",
                ))
    else:
        values.append(_report_missing_value("table_a_2_prior_policy_non_cold_deltas"))
    return values


def _report_gates_22():
    gates = []
    paired = globals().get("paired_summary")
    if _report_is_dataframe(paired) and "population" in paired.columns:
        non_cold = paired.loc[paired["population"].astype(str).eq("non-cold")]
        observed_n = _report_frame_records(non_cold, limit=20, columns=["contrast", "population", "n_cases", "n_users"])
    else:
        observed_n = None
    gates.append(_report_gate("non_cold_n", observed_n, "non-cold n recorded in paired_summary", False, None if observed_n is not None else "not_available_in_notebook"))
    model_reuse = globals().get("model_reuse_decision")
    fold_qc = globals().get("fold_qc")
    reuse_observed = {
        "model_reuse_decision": _report_frame_records(model_reuse, limit=20),
        "fold_qc": _report_frame_records(fold_qc, limit=20),
        "canonical_fold_equivalence": globals().get("canonical_fold_equivalence"),
        "canonical_param_equivalence": globals().get("canonical_param_equivalence"),
        "canonical_preprocessing_equivalence": globals().get("canonical_preprocessing_equivalence"),
    }
    gates.append(_report_gate("nb11_fold_hyperparameter_reuse", reuse_observed, "NB11/P2-Q fold and hyperparameter reuse diagnostics are recorded", _report_frame_failed(model_reuse) or _report_frame_failed(fold_qc)))
    seed = globals().get("RANDOM_SEED")
    gates.append(_report_gate("seed", seed, "42", seed not in (None, 42), None if seed is not None else "not_available_in_notebook"))
    return gates


def _report_values_26():
    values = []
    frame = globals().get("strong_weak_difference_in_delta")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            values.append(_report_value(
                claim_id=f"section7_1_strong_minus_weak:{row.get('reranker_family')}|{row.get('comparison_id')}",
                value=row.get("strong_minus_weak_delta"),
                ci=_report_ci(row, ["bootstrap_ci_95_lower", "ci_low"], ["bootstrap_ci_95_upper", "ci_high"]),
                p=None,
                n={"strong_case_count": row.get("strong_case_count"), "weak_case_count": row.get("weak_case_count")},
                source_file=_report_path_from_maps("strong_weak_difference_in_delta"),
                aggregation="existing independent strong-minus-weak bootstrap summary",
            ))
    else:
        values.append(_report_missing_value("section7_1_strong_minus_weak_delta_ci"))
    return values


def _report_gates_26():
    gates = []
    coverage = globals().get("regime_pair_coverage_qc")
    gates.append(_report_gate(
        "regime_pair_coverage",
        _report_frame_records(coverage, limit=80),
        "coverage_status PASS for each method x contrast x regime row",
        _report_frame_failed(coverage, status_columns=("coverage_status",)),
        None if _report_is_dataframe(coverage) else "not_available_in_notebook",
    ))
    strong_weak = globals().get("strong_weak_difference_in_delta")
    if _report_is_dataframe(strong_weak):
        observed_overlap = _report_frame_records(strong_weak, limit=30, columns=["reranker_family", "comparison_id", "strong_weak_case_overlap_count", "strong_weak_user_overlap_count"])
        self_flag = False
        for column in ["strong_weak_case_overlap_count", "strong_weak_user_overlap_count"]:
            if column in strong_weak.columns and _report_pd.to_numeric(strong_weak[column], errors="coerce").ne(0).any():
                self_flag = True
    else:
        observed_overlap = None
        self_flag = False
    gates.append(_report_gate("strong_weak_overlap_counts", observed_overlap, "case and user overlap counts equal 0", self_flag, None if observed_overlap is not None else "not_available_in_notebook"))
    return gates


def _report_values_28():
    values = []
    frame = globals().get("table_6_4")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            key = "|".join(str(row.get(column)) for column in ["category_label", "reranker_family", "stage_condition", "candidate_pool_depth"] if column in row)
            values.append(_report_value(
                claim_id=f"table6_4_online_ms_per_query:{key}",
                value=row.get("online_ms_per_query"),
                ci=None,
                p=None,
                n=None,
                source_file=_report_path_from_maps("table_6_4"),
                aggregation="existing online_ms_per_query from table_6_4",
            ))
            values.append(_report_value(
                claim_id=f"table6_4_hardware:{key}",
                value=row.get("compute_device"),
                ci=None,
                p=None,
                n=None,
                source_file=_report_path_from_maps("table_6_4"),
                aggregation="hardware context from existing runtime table/source manifest",
            ))
    else:
        values.append(_report_missing_value("table6_4_method_depth_ms_per_query_and_hardware"))
    return values


def _report_gates_28():
    gates = []
    runtime_qc = globals().get("runtime_qc")
    four_checks = [
        "query_denominator_present",
        "hardware_fields_present",
        "pipeline_stage1_runtime_complete",
        "offline_components_not_in_online_latency",
    ]
    if _report_is_dataframe(runtime_qc) and "check" in runtime_qc.columns:
        raw_four = runtime_qc.loc[runtime_qc["check"].astype(str).isin(four_checks)].copy()
        if raw_four.empty:
            raw_four = runtime_qc.copy()
        observed_qc = _report_frame_records(raw_four, limit=20)
        self_flag = _report_frame_failed(raw_four)
    else:
        observed_qc = None
        self_flag = False
    gates.append(_report_gate("runtime_validation_qc", observed_qc, "four raw runtime validation QC items recorded", self_flag, None if observed_qc is not None else "not_available_in_notebook"))
    run = globals().get("run_manifest", {}) if isinstance(globals().get("run_manifest", {}), dict) else {}
    mode = run.get("mode")
    gates.append(_report_gate("aggregation_only", mode, "aggregation_only_no_retrain", mode not in (None, "aggregation_only_no_retrain"), None if mode is not None else "not_available_in_notebook"))
    separation = {
        "online_definition": run.get("online_definition"),
        "offline_excluded": run.get("offline_excluded"),
        "offline_gate": next((row for row in (observed_qc or []) if row.get("check") == "offline_components_not_in_online_latency"), None),
    }
    gates.append(_report_gate("online_offline_separation", separation, "online latency excludes offline components", False if separation["offline_gate"] is not None else True, None if separation["offline_gate"] is not None else "not_available_in_notebook"))
    return gates


def _report_kind():
    name = _report_notebook.lower()
    if name.startswith("15_") or "stage_allocation_summary" in name:
        return "15"
    if name.startswith("18_") or "lightgbm_heldout_interpretation" in name:
        return "18"
    if name.startswith("19_") or "transformer_heldout_interpretation" in name:
        return "19"
    if name.startswith("20_") or "category_schema_audit" in name:
        return "20"
    if name.startswith("22_") or "prior_policy_ablation_lightgbm" in name:
        return "22"
    if name.startswith("26_") or "regime_effect_summary" in name:
        return "26"
    if name.startswith("28_") or "runtime_efficiency_summary" in name:
        return "28"
    return "unknown"


_REPORT_KIND = _report_kind()
_REPORT_SERVED = {
    "15": ["section6_condition_means", "fig6_3"],
    "18": ["fig7_1"],
    "19": ["table7_1", "section7_3"],
    "20": ["table3_e1", "appendix3_e"],
    "22": ["section7_2", "table_a_2"],
    "26": ["section7_1"],
    "28": ["table6_4"],
}.get(_REPORT_KIND, [])
_REPORT_VALUE_BUILDERS = {
    "15": _report_values_15,
    "18": _report_values_18,
    "19": _report_values_19,
    "20": _report_values_20,
    "22": _report_values_22,
    "26": _report_values_26,
    "28": _report_values_28,
}
_REPORT_GATE_BUILDERS = {
    "15": _report_gates_15,
    "18": _report_gates_18,
    "19": _report_gates_19,
    "20": _report_gates_20,
    "22": _report_gates_22,
    "26": _report_gates_26,
    "28": _report_gates_28,
}


def _report_collect_inputs():
    records = []
    seen = set()

    def add_path(path, sha=None):
        if _report_is_missing(path):
            return
        text = str(path).strip()
        if not text:
            return
        candidate = Path(text)
        if _report_is_inside(candidate, _report_output_dir()):
            return
        key = str(candidate)
        if key in seen:
            return
        seen.add(key)
        records.append({"path": key, "sha256": _report_jsonable(sha if sha is not None else _report_file_sha256(candidate, allow_heavy=False))})

    def looks_like_path(text):
        suffix = Path(str(text)).suffix.lower()
        return suffix in {".json", ".csv", ".parquet", ".txt", ".yaml", ".yml", ".ipynb", ".pkl", ".pickle", ".joblib", ".pt", ".pth", ".bin"}

    def walk(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                key_text = str(key).lower()
                if isinstance(value, (str, Path)) and (key_text.endswith("path") or key_text.endswith("paths") or looks_like_path(value)):
                    sha = None
                    if key_text.endswith("path"):
                        sha = obj.get(str(key).replace("path", "sha256")) or obj.get(str(key).replace("_path", "_sha256"))
                    add_path(value, sha)
                else:
                    walk(value)
        elif isinstance(obj, (list, tuple, set)):
            for value in obj:
                walk(value)
        elif isinstance(obj, (str, Path)) and looks_like_path(obj):
            add_path(obj)

    for obj_name in ["pipeline_manifest", "run_manifest", "analysis_manifest", "manifest"]:
        obj = globals().get(obj_name)
        if isinstance(obj, dict):
            walk(obj)

    inventory = globals().get("runtime_inventory")
    if _report_is_dataframe(inventory):
        for _, row in inventory.iterrows():
            for path_col in ["source_path", "source_manifest", "manifest_path", "path"]:
                if path_col in inventory.columns:
                    sha = None
                    for sha_col in [path_col.replace("path", "sha256"), "source_sha256", "sha256"]:
                        if sha_col in inventory.columns:
                            sha = row.get(sha_col)
                            break
                    add_path(row.get(path_col), sha)

    for name, value in list(globals().items()):
        if not name.endswith("_PATH"):
            continue
        if name.startswith(("OUTPUT", "PREDICTIONS", "PER_CASE", "DELTAS", "PAIRED_SUMMARY", "POPULATION_RESULTS", "PROFILE_DIAGNOSTICS", "SHUFFLE_ASSIGNMENT", "FALLBACK_QC", "FEATURE_CONTRACT", "MODEL_REUSE", "FOLD_QC", "LEAKAGE_QC", "COVERAGE", "RETENTION", "FEATURE_IMPORTANCE", "QC_SUMMARY", "MANIFEST")):
            if _report_is_inside(value, _report_output_dir()):
                continue
        add_path(value)

    return records


_report_run_utc = datetime.now(timezone.utc).isoformat()
_report_values = _REPORT_VALUE_BUILDERS.get(_REPORT_KIND, lambda: [_report_missing_value("requested_values")])()
_report_gates = _REPORT_GATE_BUILDERS.get(_REPORT_KIND, lambda: [_report_gate("requested_gates", None, "not_available_in_notebook", False, "not_available_in_notebook")])()
_report_lineage = {
    "notebook": _report_notebook,
    "category": _report_jsonable(_report_category),
    "run_utc": _report_run_utc,
    "code_sha": _report_file_sha256(_report_nb_path, allow_heavy=True) if _report_nb_path is not None else None,
    "inputs": _report_collect_inputs(),
}


def _report_md_cell(value):
    text = "" if value is None else (_report_jsonable(value))
    if isinstance(text, (dict, list)):
        text = json.dumps(text, ensure_ascii=False, sort_keys=True)
    text = str(text)
    return text.replace("|", "\\|").replace("\n", "<br>")


def _report_markdown(values, gates, lineage):
    lines = []
    lines.append("# Identity & lineage")
    lines.append(f"- notebook: {_report_md_cell(lineage.get('notebook'))}")
    lines.append(f"- category: {_report_md_cell(lineage.get('category'))}")
    lines.append(f"- run_utc: {_report_md_cell(lineage.get('run_utc'))}")
    lines.append(f"- code_sha: {_report_md_cell(lineage.get('code_sha'))}")
    lines.append(f"- input_count: {len(lineage.get('inputs', []))}")
    lines.append("")
    lines.append("# Served thesis elements")
    if _REPORT_SERVED:
        for served_id in _REPORT_SERVED:
            lines.append(f"- {_report_md_cell(served_id)}")
    else:
        lines.append("- not_available_in_notebook")
    lines.append("")
    lines.append("# Computed headline values")
    value_columns = ["claim_id", "value", "ci", "p", "n", "source_file", "aggregation", "thesis_value"]
    lines.append("| " + " | ".join(value_columns) + " |")
    lines.append("| " + " | ".join(["---"] * len(value_columns)) + " |")
    for record in values:
        row = dict(record)
        row["thesis_value"] = ""
        lines.append("| " + " | ".join(_report_md_cell(row.get(column)) for column in value_columns) + " |")
    lines.append("")
    lines.append("# QC gates")
    gate_columns = ["gate_id", "observed", "expected_contract", "self_flag"]
    lines.append("| " + " | ".join(gate_columns) + " |")
    lines.append("| " + " | ".join(["---"] * len(gate_columns)) + " |")
    for record in gates:
        lines.append("| " + " | ".join(_report_md_cell(record.get(column)) for column in gate_columns) + " |")
    lines.append("")
    lines.append("# Self-detected anomalies")
    anomalies = []
    for record in gates:
        if record.get("self_flag") is True:
            anomalies.append(f"- gate_self_flag: {_report_md_cell(record.get('gate_id'))}")
        if record.get("note"):
            anomalies.append(f"- gate_note: {_report_md_cell(record.get('gate_id'))}: {_report_md_cell(record.get('note'))}")
    for record in values:
        if record.get("note"):
            anomalies.append(f"- value_note: {_report_md_cell(record.get('claim_id'))}: {_report_md_cell(record.get('note'))}")
    lines.extend(anomalies if anomalies else ["- none"])
    lines.append("")
    return "\n".join(lines)


_lineage_path = _report_dir / "lineage.json"
_values_path = _report_dir / "report_values.json"
_gates_path = _report_dir / "qc_gates.json"
_markdown_path = _report_dir / "verification_report.md"
_lineage_path.write_text(json.dumps(_report_lineage, ensure_ascii=False, indent=2), encoding="utf-8")
_values_path.write_text(json.dumps(_report_values, ensure_ascii=False, indent=2), encoding="utf-8")
_gates_path.write_text(json.dumps(_report_gates, ensure_ascii=False, indent=2), encoding="utf-8")
_markdown_path.write_text(_report_markdown(_report_values, _report_gates, _report_lineage), encoding="utf-8")
for _written_path in [_lineage_path, _values_path, _gates_path, _markdown_path]:
    print(_written_path)


/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/stage_allocation_summary/report/lineage.json
/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/stage_allocation_summary/report/report_values.json
/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/stage_allocation_summary/report/qc_gates.json
/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/stage_allocation_summary/report/verification_report.md
